# 04 Financial data (WRDS)

Compustat, CRSP (CIZ), I/B/E/S and the CRSP-Compustat link table; fiscal-year alignment, winsorisation and merge with the NLP features. Outputs: `data/processed/financials_2017_2025.parquet`, `data/processed/analysis_panel_2018_2025.parquet`, `data/intermediate/crsp_daily_returns_around_filing.parquet`. Variables in Appendix X. Requires a WRDS account.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
import wrds
from pathlib import Path
from scipy import stats

# ── Paths ──────────────────────────────────────────────────────────────────
DATA_OUT    = Path("../data/processed")
DATA_INTER     = Path("../data/intermediate")

DATA_OUT.mkdir(parents=True, exist_ok=True)
DATA_INTER.mkdir(parents=True, exist_ok=True)

# ── Parameters ──────────────────────────────────────────────────────────────
# Set TEST_MODE = True for a fast first run (FY2018 only, ~150 firms).
# Set TEST_MODE = False for the full 2017–2025 pull.
TEST_MODE   = False

YEAR_START  = 2018 if TEST_MODE else 2017
YEAR_END    = 2018 if TEST_MODE else 2025
BETA_WINDOW = 36

# CRSP must start BETA_WINDOW months before earliest fiscal year end.
# Full run: earliest FYE = Dec 2017 → need Jan 2015 → use Dec 2014.
# Test run: earliest FYE = Dec 2018 → need Jan 2016 → use Dec 2015.
CRSP_START  = "2015-12-01" if TEST_MODE else "2014-12-01"
CRSP_END    = "2018-12-31" if TEST_MODE else "2025-12-31"

OUT_SUFFIX  = "test_2018" if TEST_MODE else "2017_2025"

print(f"{'TEST MODE' if TEST_MODE else 'FULL RUN'}")
print(f"  Funda   : fyear {YEAR_START}–{YEAR_END}")
print(f"  CRSP    : {CRSP_START} → {CRSP_END}")
print(f"  Output  : financials_{OUT_SUFFIX}.parquet")

FULL RUN
  Funda   : fyear 2017–2025
  CRSP    : 2014-12-01 → 2025-12-31
  Output  : financials_2017_2025.parquet


## 1 Firm universe and WRDS connection

In [9]:
# Load the firm universe ────────────────────────────────────────────────────
universe = pd.read_csv("../data/SP1500_CIK.csv", usecols=["Symbol", "company_name", "Index", "cik"])
universe = universe.dropna(subset=["cik"]).copy()
universe["cik"] = universe["cik"].astype(int)
# Compustat stores CIK as VARCHAR(10) zero-padded
universe["cik_str"] = universe["cik"].apply(lambda x: str(x).zfill(10))

print(f"Firm universe: {len(universe):,} firms")
print(universe["Index"].value_counts().to_string())
universe.head(3)

Firm universe: 1,498 firms
Index
S&P 600    597
S&P 500    503
S&P 400    398


,Symbol,company_name,Index,cik,cik_str
0,A,Agilent Technologies,S&P 500,1090872,0001090872
1,AAPL,Apple Inc,S&P 500,320193,0000320193
2,ABBV,Abbvie Inc,S&P 500,1551152,0001551152


In [14]:
# Connect to WRDS ─────────────────────────────────────────────────────────
# First run: prompts for username/password and saves credentials to ~/.pgpass
conn = wrds.Connection(wrds_username='soerenssch')
print("Connected to WRDS.")

OperationalError: (psycopg2.OperationalError) connection to server at "wrds-pgdata.wharton.upenn.edu" (165.123.60.118), port 9737 failed: FATAL:  PAM authentication failed for user "soerenssch"

(Background on this error at: https://sqlalche.me/e/20/e3q8)

## 2 Identifier bridge: CIK -> GVKEY -> PERMNO

In [5]:
# ── 2a. CIK → GVKEY via comp.company ───────────────────────────────────────
cik_sql = "','".join(universe["cik_str"])

comp_company = conn.raw_sql(f"""
    SELECT gvkey, cik, conm, sic, naics, fic, state, stko
    FROM comp.company
    WHERE cik IN ('{cik_sql}')
""", date_cols=[])

comp_company["cik_int"] = comp_company["cik"].str.lstrip("0").replace("", "0").astype(int)

print(f"Matched {len(comp_company):,} / {len(universe):,} firms via CIK → GVKEY")

# Firms that did not match (usually SPACs, foreign private issuers, or CIK mismatches)
unmatched = universe[~universe["cik"].isin(comp_company["cik_int"])]
print(f"Unmatched: {len(unmatched)} firms")
if len(unmatched) > 0:
    print(unmatched[["Symbol","company_name","cik"]].to_string(index=False))

Matched 1,491 / 1,498 firms via CIK → GVKEY
Unmatched: 1 firms
Symbol        company_name     cik
  PNFP Pinnacle Finl Ptnrs 2082866


In [6]:
# ── 2b. GVKEY → PERMNO via CRSP-Compustat merged link ──────────────────────
gvkey_list = comp_company["gvkey"].tolist()
gvkey_sql  = "','".join(gvkey_list)

ccm_link = conn.raw_sql(f"""
    SELECT gvkey,
           lpermno  AS permno,
           lpermco  AS permco,
           linktype, linkprim,
           linkdt, linkenddt
    FROM crsp.ccmxpf_linktable
    WHERE gvkey IN ('{gvkey_sql}')
      AND linktype IN ('LU', 'LC')
      AND linkprim IN ('P', 'C')
""", date_cols=["linkdt", "linkenddt"])

ccm_link["linkenddt"] = ccm_link["linkenddt"].fillna(pd.Timestamp("2099-12-31"))

print(f"CCM link rows: {len(ccm_link):,}")
print(f"Unique GVKEYs with PERMNO: {ccm_link['gvkey'].nunique():,}")
print(f"Unique PERMNOs: {ccm_link['permno'].nunique():,}")

# ── 2c. Build master bridge ─────────────────────────────────────────────────
# universe has 'cik' (int); comp_company also has 'cik' (str).
# Merge with suffixes to avoid collision, then drop the Compustat copy.
bridge = (universe
    .merge(
        comp_company[["gvkey","cik","sic","naics","fic","state"]],
        left_on="cik_str", right_on="cik",
        how="left",
        suffixes=("", "_comp")   # universe cik stays 'cik'; comp copy becomes 'cik_comp'
    )
    .drop(columns=["cik_comp"])
    .merge(ccm_link[["gvkey","permno","linkdt","linkenddt"]],
           on="gvkey", how="left")
)

print(f"\nBridge rows: {len(bridge):,}")
print(f"  with GVKEY : {bridge['gvkey'].notna().sum():,}")
print(f"  with PERMNO: {bridge['permno'].notna().sum():,}")

bridge.to_parquet(DATA_INTER / "identifier_bridge.parquet", index=False)
bridge.head(3)

CCM link rows: 1,814
Unique GVKEYs with PERMNO: 1,489
Unique PERMNOs: 1,623

Bridge rows: 1,825
  with GVKEY : 1,824
  with PERMNO: 1,822


,Symbol,company_name,Index,cik,cik_str,gvkey,sic,naics,fic,state,permno,linkdt,linkenddt
0,A,Agilent Technologies,S&P 500,1090872,0001090872,126554,3826,334516,USA,CA,87432.0,1999-11-18,2099-12-31
1,AAPL,Apple Inc,S&P 500,320193,0000320193,001690,3663,334220,USA,CA,14593.0,1980-12-12,2099-12-31
2,ABBV,Abbvie Inc,S&P 500,1551152,0001551152,016101,2836,325414,USA,IL,13721.0,2013-01-02,2099-12-31


## 3 Compustat annual fundamentals (`comp.funda`)

In [7]:
funda_raw = conn.raw_sql(f"""
    SELECT
        gvkey, cik, datadate, fyear, fyr, conm, sich, naicsh,
        -- Size / revenue
        at, sale, revt, cogs, xsga,
        -- Profitability
        ni, ib, oibdp, oiadp, pi,
        -- Leverage
        dltt, dlc, dd1, lt, ceq, seq,
        -- Liquidity
        act, lct, che, rect, invt,
        -- Investment / assets
        capx, xrd, dp, ppent,
        -- International
        pifo, txfo,
        -- Equity / market
        csho, prcc_f, ajex, mkvalt, dvt,
        -- Credit
        dd
    FROM comp.funda
    WHERE gvkey IN ('{gvkey_sql}')
      AND fyear BETWEEN {YEAR_START} AND {YEAR_END}
      AND indfmt  = 'INDL'
      AND datafmt = 'STD'
      AND popsrc  = 'D'
      AND consol  = 'C'
""", date_cols=["datadate"])

# Rename naicsh → naics for consistency downstream
funda_raw = funda_raw.rename(columns={"naicsh": "naics"})

print(f"Compustat funda rows: {len(funda_raw):,}")
print(f"Unique GVKEYs: {funda_raw['gvkey'].nunique():,}  |  "
      f"Fiscal years: {sorted(funda_raw['fyear'].dropna().astype(int).unique())}")

funda_raw.to_parquet(DATA_INTER / "funda_raw.parquet", index=False)
funda_raw.head(3)

Compustat funda rows: 13,053
Unique GVKEYs: 1,491  |  Fiscal years: [np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


,gvkey,cik,datadate,fyear,fyr,conm,sich,naics,at,sale,...,dp,ppent,pifo,txfo,csho,prcc_f,ajex,mkvalt,dvt,dd
0,001004,0000001750,2018-05-31,2017,5,AAR CORP,5080,423860,1524.7,1748.3,...,40.5,316.6,18.5,1.7,34.716,44.69,1.0,1551.458,10.3,25.0
1,001004,0000001750,2019-05-31,2018,5,AAR CORP,5080,423860,1517.2,2051.8,...,42.8,348.8,20.1,5.0,34.788,30.09,1.0,1046.7709,10.5,0.0
2,001004,0000001750,2020-05-31,2019,5,AAR CORP,5080,423860,2079.0,2089.3,...,43.7,437.1,6.8,2.8,35.097,20.17,1.0,707.9065,10.7,0.0


In [8]:
# ── FY2025 coverage check ────────────────────────────────────────────────────
# Compustat has a 4–8 week lag after filing. Since it is currently early 2026,
# FY2025 data may still be incomplete. Skip this check in test mode.

coverage_by_year = funda_raw.groupby("fyear")["gvkey"].nunique().sort_index()
print("Firms with Compustat data per fiscal year:")
print(coverage_by_year.to_string())

if not TEST_MODE:
    fy2025_count = coverage_by_year.get(2025, 0)
    fy2024_count = coverage_by_year.get(2024, 0)
    if fy2024_count == 0:
        print("\n⚠  No FY2024 data found — cannot assess FY2025 coverage.")
    elif fy2025_count < 0.7 * fy2024_count:
        print(f"\n⚠  FY2025 coverage is low ({fy2025_count} vs {fy2024_count} in 2024).")
        print("   Consider re-running in April/May 2026 for fuller coverage.")
    else:
        print(f"\n✓  FY2025 looks complete ({fy2025_count} firms, {fy2025_count/fy2024_count:.0%} of 2024).")
else:
    print("\n[TEST MODE] FY2025 coverage check skipped.")

Firms with Compustat data per fiscal year:
fyear
2017    1403
2018    1429
2019    1453
2020    1466
2021    1480
2022    1490
2023    1491
2024    1491
2025    1349

✓  FY2025 looks complete (1349 firms, 90% of 2024).


## 4 CRSP market data: market capitalisation, annual return, beta

In [9]:
# Unique PERMNOs from the link table (already filtered to our GVKEYs)
permno_list = ccm_link["permno"].dropna().astype(int).unique().tolist()
permno_sql  = ",".join(map(str, permno_list))

# ── 4a. Pull monthly stock file ─────────────────────────────────────────────
crsp_msf = conn.raw_sql(f"""
    SELECT permno, date,
           ABS(prc)           AS prc,
           ret,
           retx,              -- return excluding dividends (for price-only return)
           shrout             -- shares outstanding (thousands)
    FROM crsp.msf
    WHERE permno IN ({permno_sql})
      AND date BETWEEN '{CRSP_START}' AND '{CRSP_END}'
""", date_cols=["date"])

crsp_msf["mktcap"] = crsp_msf["prc"] * crsp_msf["shrout"]   # in $thousands

print(f"CRSP monthly rows: {len(crsp_msf):,}")
print(f"Unique PERMNOs: {crsp_msf['permno'].nunique():,}")

crsp_msf.to_parquet(DATA_INTER / "crsp_msf.parquet", index=False)

CRSP monthly rows: 169,268
Unique PERMNOs: 1,520


In [10]:
# ── 4b. Pull CRSP value-weighted market return for beta ─────────────────────
crsp_msi = conn.raw_sql(f"""
    SELECT date, vwretd AS mkt_ret
    FROM crsp.msi
    WHERE date BETWEEN '{CRSP_START}' AND '{CRSP_END}'
""", date_cols=["date"])

crsp_msi = crsp_msi.sort_values("date").reset_index(drop=True)
print(f"Market return months: {len(crsp_msi):,}")

Market return months: 121


In [11]:
# ── 4c. Align CRSP to Compustat fiscal year ends ────────────────────────────
#
# For each funda row (gvkey, datadate) we need the PERMNO valid at that date.
# Use the CCM link date range: linkdt <= datadate <= linkenddt.

def get_permno_for_date(gvkey, date, link_df):
    """Return PERMNO valid at `date` for a given GVKEY."""
    mask = (
        (link_df["gvkey"] == gvkey) &
        (link_df["linkdt"]    <= date) &
        (link_df["linkenddt"] >= date)
    )
    rows = link_df.loc[mask, "permno"]
    return rows.iloc[0] if len(rows) > 0 else np.nan

# Merge on gvkey first, then filter by date range (vectorised)
funda_link = funda_raw[["gvkey","datadate","fyear"]].drop_duplicates()
funda_link = funda_link.merge(
    ccm_link[["gvkey","permno","linkdt","linkenddt"]], on="gvkey", how="left"
)
funda_link = funda_link[
    (funda_link["linkdt"]    <= funda_link["datadate"]) &
    (funda_link["linkenddt"] >= funda_link["datadate"])
].drop_duplicates(subset=["gvkey","datadate"])

print(f"Funda rows matched to PERMNO: {len(funda_link):,} / {len(funda_raw):,}")

Funda rows matched to PERMNO: 12,680 / 13,053


In [12]:
# ── 4d. Compute market cap, annual return, and beta per firm-year ────────────

# Merge CRSP with market return for beta computation
crsp_merged = crsp_msf.merge(crsp_msi[["date","mkt_ret"]], on="date", how="left")
crsp_merged = crsp_merged.sort_values(["permno","date"])

# Month-end date aligned to fiscal year end (floor to month)
crsp_merged["ym"] = crsp_merged["date"].dt.to_period("M")

def compute_crsp_annual(row, crsp_df, mkt_df, beta_window=36):
    """
    For a single funda observation (permno, datadate), compute:
    - mktcap   : market cap at fiscal year end ($thousands)
    - ret_ann  : buy-and-hold return over fiscal year (12 months)
    - beta     : market beta from OLS, beta_window months ending at datadate
    """
    permno   = row["permno"]
    end_date = row["datadate"]
    if pd.isna(permno) or pd.isna(end_date):
        return pd.Series({"mktcap": np.nan, "ret_ann": np.nan, "beta": np.nan})

    end_ym   = pd.Period(end_date, freq="M")
    start_ym = end_ym - 11            # 12-month window for annual return
    beta_ym  = end_ym - (beta_window - 1)

    stock = crsp_df[crsp_df["permno"] == permno].copy()

    # Market cap at fiscal year end
    end_row = stock[stock["ym"] == end_ym]
    mktcap  = end_row["mktcap"].iloc[0] if len(end_row) > 0 else np.nan

    # Annual return: compound monthly ret over [start_ym, end_ym]
    annual_slice = stock[(stock["ym"] >= start_ym) & (stock["ym"] <= end_ym)]["ret"].dropna()
    ret_ann = (1 + annual_slice).prod() - 1 if len(annual_slice) >= 6 else np.nan

    # Beta: OLS over [beta_ym, end_ym]
    beta_slice = stock[(stock["ym"] >= beta_ym) & (stock["ym"] <= end_ym)][["ym","ret"]].dropna()
    beta_slice = beta_slice.merge(
        crsp_msi.assign(ym=crsp_msi["date"].dt.to_period("M"))[["ym","mkt_ret"]],
        on="ym", how="inner"
    ).dropna()
    if len(beta_slice) >= 12:
        slope, _, _, _, _ = stats.linregress(beta_slice["mkt_ret"], beta_slice["ret"])
        beta = slope
    else:
        beta = np.nan

    return pd.Series({"mktcap": mktcap, "ret_ann": ret_ann, "beta": beta})

# Apply — this takes a few minutes for ~15k rows
print("Computing CRSP market variables (may take a few minutes)...")
crsp_annual = funda_link.apply(
    lambda row: compute_crsp_annual(row, crsp_merged, crsp_msi, BETA_WINDOW), axis=1
)
crsp_annual = pd.concat([funda_link[["gvkey","datadate","permno"]], crsp_annual], axis=1)

print(f"\nCRSP annual rows: {len(crsp_annual):,}")
print(crsp_annual[["mktcap","ret_ann","beta"]].describe().round(3))

crsp_annual.to_parquet(DATA_INTER / "crsp_annual.parquet", index=False)

Computing CRSP market variables (may take a few minutes)...

CRSP annual rows: 12,680
         ret_ann       beta
count  11295.000  12473.000
mean       0.172      1.190
std        0.933      0.656
min       -0.980     -3.412
25%       -0.096      0.795
50%        0.111      1.130
75%        0.337      1.508
max       83.635      7.816


## 5 Geographic segments (`comp.segg`)

Not accessible in the final pull; the foreign income share `pifo / pi` is used instead.

In [13]:
# ── 5a. Diagnostic: check which segment tables are accessible ───────────────
SEGG_CANDIDATES = ["comp.segg", "compa.segg", "comp_segments.segg"]
SEGG_TABLE = None

for tbl in SEGG_CANDIDATES:
    try:
        conn.raw_sql(f"SELECT 1 FROM {tbl} LIMIT 1", date_cols=[])
        SEGG_TABLE = tbl
        print(f"✓  Found segments table: {tbl}")
        break
    except Exception:
        print(f"✗  Not accessible: {tbl}")

if SEGG_TABLE is None:
    print("\n⚠  No geographic segments table found.")
    print("   Falling back to pifo/pi (foreign pre-tax income) from comp.funda.")
    print("   Request 'Compustat Segments' access via your institution's WRDS portal if needed.")

✗  Not accessible: comp.segg
✗  Not accessible: compa.segg
✗  Not accessible: comp_segments.segg

⚠  No geographic segments table found.
   Falling back to pifo/pi (foreign pre-tax income) from comp.funda.
   Request 'Compustat Segments' access via your institution's WRDS portal if needed.


In [14]:
# ── 5b. Pull & compute geographic features ──────────────────────────────────
if SEGG_TABLE is not None:
    segg_raw = conn.raw_sql(f"""
        SELECT gvkey, datadate, srcdate,
               sid, snms, stype,
               geoent, revts, ois, assets
        FROM {SEGG_TABLE}
        WHERE gvkey IN ('{gvkey_sql}')
          AND datadate BETWEEN '{YEAR_START}-01-01' AND '{YEAR_END}-12-31'
          AND stype = 'GEOSEG'
    """, date_cols=["datadate", "srcdate"])

    print(f"Geographic segment rows: {len(segg_raw):,}")
    print(f"Unique GVKEYs with geo segments: {segg_raw['gvkey'].nunique():,}")
    print(f"\nTop geoent codes:\n{segg_raw['geoent'].value_counts().head(10).to_string()}")
    segg_raw.to_parquet(DATA_INTER / "segg_raw.parquet", index=False)

    # ── Compute foreign revenue share ─────────────────────────────────────
    segg = segg_raw.copy()
    segg["is_domestic"] = segg["geoent"] == "DOM"
    segg["is_total"]    = segg["geoent"] == "TOTAL"

    def geo_features(df):
        total_row = df[df["is_total"]]["revts"].sum()
        dom_row   = df[df["is_domestic"]]["revts"].sum()
        seg_rows  = df[~df["is_total"]]["revts"].sum()
        n_foreign = int((~df["is_total"] & ~df["is_domestic"]).sum())
        total_rev   = total_row if total_row > 0 else seg_rows
        foreign_rev = total_rev - dom_row if total_rev > 0 else np.nan
        return pd.Series({
            "seg_total_rev"    : total_rev,
            "seg_domestic_rev" : dom_row,
            "seg_foreign_rev"  : foreign_rev,
            "foreign_rev_share": foreign_rev / total_rev if (total_rev > 0 and not np.isnan(foreign_rev)) else np.nan,
            "n_geo_segments"   : n_foreign,
        })

    geo_annual = (segg
        .groupby(["gvkey","datadate"])
        .apply(geo_features, include_groups=False)
        .reset_index()
    )
    print(f"\nGeographic features rows: {len(geo_annual):,}")
    print(geo_annual[["foreign_rev_share","n_geo_segments"]].describe().round(3))
    geo_annual.to_parquet(DATA_INTER / "geo_annual.parquet", index=False)

else:
    # Fallback: derive international exposure from funda (pifo / pi)
    print("Building international exposure proxy from comp.funda (pifo/pi)...")
    geo_annual = funda_raw[["gvkey","datadate","pifo","pi"]].copy()
    geo_annual["foreign_rev_share"] = np.where(
        (geo_annual["pi"].notna()) & (geo_annual["pi"] != 0) & (geo_annual["pifo"].notna()),
        (geo_annual["pifo"] / geo_annual["pi"]).clip(-1, 2),
        np.nan
    )
    geo_annual["n_geo_segments"] = np.nan
    geo_annual["seg_total_rev"]  = np.nan
    geo_annual["seg_domestic_rev"] = np.nan
    geo_annual["seg_foreign_rev"]  = np.nan
    geo_annual = geo_annual.drop(columns=["pifo","pi"])
    print(f"Fallback geo proxy rows: {len(geo_annual):,}")
    print(f"  Non-null foreign_rev_share: {geo_annual['foreign_rev_share'].notna().sum():,}")
    geo_annual.to_parquet(DATA_INTER / "geo_annual.parquet", index=False)

Building international exposure proxy from comp.funda (pifo/pi)...
Fallback geo proxy rows: 13,053
  Non-null foreign_rev_share: 8,117


## 6 Institutional ownership (`tfn.s34`)

In [15]:
# ── 6a. Get 8-digit CUSIPs for our PERMNOs via crsp.stocknames ──────────────
stocknames = conn.raw_sql(f"""
    SELECT permno, ncusip, cusip, namedt, nameenddt
    FROM crsp.stocknames
    WHERE permno IN ({permno_sql})
""", date_cols=["namedt", "nameenddt"])

stocknames = stocknames.rename(columns={"nameenddt": "nameendt"})
stocknames["nameendt"] = stocknames["nameendt"].fillna(pd.Timestamp("2099-12-31"))

cusip8_list = stocknames["ncusip"].dropna().unique().tolist()
cusip8_sql  = "','".join(cusip8_list)

print(f"Unique 8-digit CUSIPs: {len(cusip8_list):,}")

Unique 8-digit CUSIPs: 2,472


In [16]:
# ── 6b. Check 13F access, then pull ─────────────────────────────────────────
TFN_AVAILABLE = False
try:
    conn.raw_sql("SELECT 1 FROM tfn.s34 LIMIT 1", date_cols=[])
    TFN_AVAILABLE = True
    print("✓  tfn.s34 accessible")
except Exception as e:
    print(f"✗  tfn.s34 not accessible: {type(e).__name__}")
    print("   Skipping institutional ownership. Request 'Thomson Reuters 13F' via WRDS portal.")

if TFN_AVAILABLE:
    tfn_raw = conn.raw_sql(f"""
        SELECT rdate, cusip, SUM(shares) AS inst_shares_k
        FROM tfn.s34
        WHERE cusip IN ('{cusip8_sql}')
          AND rdate BETWEEN '{YEAR_START}-01-01' AND '{YEAR_END}-12-31'
        GROUP BY rdate, cusip
    """, date_cols=["rdate"])
    print(f"13F aggregated rows: {len(tfn_raw):,}")
else:
    tfn_raw = pd.DataFrame(columns=["rdate","cusip","inst_shares_k"])

✗  tfn.s34 not accessible: ProgrammingError
   Skipping institutional ownership. Request 'Thomson Reuters 13F' via WRDS portal.


In [17]:
# ── 6c. Link 13F CUSIP → PERMNO ─────────────────────────────────────────────
if TFN_AVAILABLE and len(tfn_raw) > 0:
    tfn_link = tfn_raw.merge(
        stocknames[["permno","ncusip","namedt","nameendt"]],
        left_on="cusip", right_on="ncusip", how="left"
    )
    tfn_link = tfn_link[
        (tfn_link["rdate"] >= tfn_link["namedt"]) &
        (tfn_link["rdate"] <= tfn_link["nameendt"])
    ].drop(columns=["ncusip","namedt","nameendt"])

    crsp_qtr = crsp_msf.copy()
    crsp_qtr["yq"] = crsp_qtr["date"].dt.to_period("Q")
    crsp_qtr = crsp_qtr.groupby(["permno","yq"]).last().reset_index()

    tfn_link["yq"] = tfn_link["rdate"].dt.to_period("Q")
    tfn_io = tfn_link.merge(crsp_qtr[["permno","yq","shrout"]], on=["permno","yq"], how="left")
    tfn_io["inst_ownership"] = (tfn_io["inst_shares_k"] / tfn_io["shrout"]).clip(0, 1)
    tfn_io["rdate_ym"] = tfn_io["rdate"].dt.to_period("M")
    print(f"13F IO rows after linkage: {len(tfn_io):,}")
    print(tfn_io["inst_ownership"].describe().round(3))
else:
    tfn_io = pd.DataFrame(columns=["permno","rdate","inst_ownership","rdate_ym"])
    print("Skipped (no 13F access).")

Skipped (no 13F access).


In [18]:
# ── 6d. Match 13F IO to Compustat fiscal year ends ──────────────────────────
funda_permno = funda_link[["gvkey","datadate","permno"]].dropna(subset=["permno"])
funda_permno["permno"] = funda_permno["permno"].astype(int)

if TFN_AVAILABLE and len(tfn_io) > 0:
    io_match = funda_permno.merge(
        tfn_io[["permno","rdate","inst_ownership"]].dropna(), on="permno", how="left"
    )
    io_match = io_match[io_match["rdate"] <= io_match["datadate"]]
    io_match["days_diff"] = (io_match["datadate"] - io_match["rdate"]).dt.days
    io_match = io_match[io_match["days_diff"] <= 183]
    io_annual = (io_match
        .sort_values("days_diff")
        .drop_duplicates(subset=["gvkey","datadate"])
        [["gvkey","datadate","inst_ownership"]]
    )
    print(f"Institutional ownership matched: {len(io_annual):,} firm-years")
    print(io_annual["inst_ownership"].describe().round(3))
else:
    # No 13F access — produce empty frame; inst_ownership will be NaN in final panel
    io_annual = pd.DataFrame(columns=["gvkey","datadate","inst_ownership"])
    print("inst_ownership set to NaN for all rows (no 13F access).")

io_annual.to_parquet(DATA_INTER / "io_annual.parquet", index=False)

inst_ownership set to NaN for all rows (no 13F access).


## 7 Analyst coverage (`ibes.statsum_epsus`)

In [19]:
# ── 7a. Link IBES ticker → PERMNO via CUSIP ─────────────────────────────────
# ibes.idsum has: ticker, cusip (8-digit), cname — no permno column.
# Match cusip → permno through crsp.stocknames (already pulled above).

IBES_AVAILABLE = False
try:
    conn.raw_sql("SELECT 1 FROM ibes.idsum LIMIT 1", date_cols=[])
    IBES_AVAILABLE = True
    print("✓  ibes.idsum accessible")
except Exception as e:
    print(f"✗  ibes.idsum not accessible: {type(e).__name__}")

if IBES_AVAILABLE:
    ibes_id_raw = conn.raw_sql(f"""
        SELECT ticker, cusip, cname
        FROM ibes.idsum
        WHERE cusip IN ('{cusip8_sql}')
    """, date_cols=[])

    # Join to stocknames to get permno (match on ncusip = 8-digit CUSIP)
    ibes_id = ibes_id_raw.merge(
        stocknames[["permno","ncusip"]].drop_duplicates(),
        left_on="cusip", right_on="ncusip", how="left"
    ).drop(columns=["ncusip"]).dropna(subset=["permno"])
    ibes_id["permno"] = ibes_id["permno"].astype(int)

    ibes_tickers  = ibes_id["ticker"].dropna().unique().tolist()
    ibes_tick_sql = "','".join(ibes_tickers)
    print(f"IBES tickers matched: {len(ibes_tickers):,}")
else:
    ibes_id       = pd.DataFrame(columns=["ticker","cusip","permno"])
    ibes_tickers  = []
    ibes_tick_sql = ""
    print("Skipping IBES (no access).")

✓  ibes.idsum accessible
IBES tickers matched: 1,569


In [20]:
# ── 7b. Pull I/B/E/S summary statistics & match to fiscal year ends ──────────
if IBES_AVAILABLE and ibes_tick_sql:
    ibes_raw = conn.raw_sql(f"""
        SELECT ticker, statpers, numest, medest, meanest, stdev, fpedats
        FROM ibes.statsum_epsus
        WHERE ticker  IN ('{ibes_tick_sql}')
          AND statpers BETWEEN '{YEAR_START}-01-01' AND '{YEAR_END}-12-31'
          AND fpi     = '1'
          AND measure = 'EPS'
    """, date_cols=["statpers","fpedats"])

    print(f"IBES raw rows: {len(ibes_raw):,}")

    ibes_with_permno = ibes_raw.merge(ibes_id[["ticker","permno"]], on="ticker", how="left").dropna(subset=["permno"])
    ibes_with_permno["permno"] = ibes_with_permno["permno"].astype(int)

    ibes_match = funda_permno.merge(
        ibes_with_permno[["permno","statpers","numest","medest","stdev"]],
        on="permno", how="left"
    )
    ibes_match = ibes_match[
        (ibes_match["statpers"] <= ibes_match["datadate"]) &
        ((ibes_match["datadate"] - ibes_match["statpers"]).dt.days <= 365)
    ]
    ibes_annual = (ibes_match
        .sort_values("statpers", ascending=False)
        .drop_duplicates(subset=["gvkey","datadate"])
        [["gvkey","datadate","numest","medest","stdev"]]
        .rename(columns={"numest":"analyst_coverage","medest":"eps_median","stdev":"eps_dispersion"})
    )
    print(f"Analyst coverage matched: {len(ibes_annual):,} firm-years")
    print(ibes_annual["analyst_coverage"].describe().round(1))
else:
    ibes_annual = pd.DataFrame(columns=["gvkey","datadate","analyst_coverage","eps_median","eps_dispersion"])
    print("analyst_coverage set to NaN for all rows (no IBES access).")

ibes_annual.to_parquet(DATA_INTER / "ibes_annual.parquet", index=False)

IBES raw rows: 152,022
Analyst coverage matched: 12,531 firm-years
count    12531.0
mean        11.3
std          7.9
min          1.0
25%          5.0
50%          9.0
75%         16.0
max         62.0
Name: analyst_coverage, dtype: Float64


## 8 Additional variables

In [21]:
# ── 8a. Sales Growth (YoY revenue growth) ────────────────────────────────────
# sales_growth = (sale_t − sale_{t−1}) / |sale_{t−1}|
# Computed within gvkey groups by shifting the sorted funda panel.
# NOTE: In TEST_MODE (single year), all values will be NaN because there is no
# FY t−1 row in funda_raw.  Set TEST_MODE = False to populate this variable.

funda_sg = (
    funda_raw[["gvkey", "datadate", "fyear", "sale"]]
    .sort_values(["gvkey", "fyear"])
    .copy()
)
funda_sg["sale_lag"] = funda_sg.groupby("gvkey")["sale"].shift(1)

valid = funda_sg["sale_lag"].notna() & (funda_sg["sale_lag"].abs() > 0.01)
funda_sg["sales_growth"] = np.where(
    valid,
    (funda_sg["sale"] - funda_sg["sale_lag"]) / funda_sg["sale_lag"].abs(),
    np.nan,
)
funda_sg["sales_growth"] = pd.to_numeric(funda_sg["sales_growth"]).clip(-1.0, 5.0)

sales_growth_annual = funda_sg[["gvkey", "datadate", "sales_growth"]]

print(f"sales_growth non-null: {sales_growth_annual['sales_growth'].notna().sum():,} "
      f"/ {len(sales_growth_annual):,}")
if TEST_MODE:
    print("[TEST MODE] All NaN expected — no FY t-1 data in funda_raw.")
print(sales_growth_annual["sales_growth"].describe().round(3))

# ── 8b. Firm Risk (rolling 5-year SD of ROA) ─────────────────────────────────
# firm_risk = rolling standard deviation of ROA over the trailing 5 fiscal years
# (min 3 non-NaN observations required).  Captures earnings volatility.
# NOTE: In TEST_MODE or short panels (< 3 years) all values will be NaN.

funda_risk = (
    funda_raw[["gvkey", "datadate", "fyear", "ni", "at"]]
    .sort_values(["gvkey", "fyear"])
    .copy()
)
funda_risk["roa_temp"] = np.where(
    funda_risk["at"].notna() & (funda_risk["at"] > 0.01),
    funda_risk["ni"] / funda_risk["at"],
    np.nan,
)
funda_risk["firm_risk"] = (
    funda_risk.groupby("gvkey")["roa_temp"]
    .transform(lambda x: x.rolling(window=5, min_periods=3).std())
)

firm_risk_annual = funda_risk[["gvkey", "datadate", "firm_risk"]]

print(f"\nfirm_risk non-null: {firm_risk_annual['firm_risk'].notna().sum():,} "
      f"/ {len(firm_risk_annual):,}")
if TEST_MODE:
    print("[TEST MODE] All NaN expected — need ≥3 years of ROA per firm.")
print(firm_risk_annual["firm_risk"].describe().round(4))

sales_growth non-null: 11,461 / 13,053
count    11461.000
mean         0.121
std          0.363
min         -1.000
25%         -0.003
50%          0.071
75%          0.166
max          5.000
Name: sales_growth, dtype: float64

firm_risk non-null: 9,988 / 13,053
count    9988.0000
mean        0.0442
std         0.0726
min         0.0001
25%         0.0110
50%         0.0245
75%         0.0491
max         1.1782
Name: firm_risk, dtype: float64


In [22]:
# ── 8c. Fama-French 48 Industry Classification ──────────────────────────────
# Maps 4-digit SIC code to the Fama-French 48 industry classification.
# Source: Ken French Data Library
#   https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/Data_Library/
# SIC priority: sich (historical, funda) → sic (comp.company, via comp_company df)
# `sich` in comp.funda is often NULL for recent fiscal years; comp.company.sic
# is the current SIC and is far more reliably populated.
# Industry 48 ("Other") is the catch-all for unclassified SIC codes.

def _build_ff48_lookup():
    """Return (codes[10000], names[10000]) arrays indexed by 4-digit SIC."""
    codes = np.full(10000, 48, dtype=np.int8)
    names = np.full(10000, "Other", dtype=object)

    def fill(lo, hi, code, name):
        codes[lo: hi + 1] = code
        names[lo: hi + 1] = name

    # (ff48_code, ff48_name, [(sic_lo, sic_hi), ...])
    # Later entries overwrite earlier ones for the same SIC (correct for FF48).
    ff48_def = [
        (1,  "Agric",  [(100,299),(700,799),(910,919),(2048,2048)]),
        (2,  "Food",   [(2000,2046),(2050,2063),(2070,2079),(2090,2095),(2098,2099)]),
        (3,  "Soda",   [(2064,2068),(2086,2087),(2096,2097)]),
        (4,  "Beer",   [(2080,2085)]),
        (5,  "Smoke",  [(2100,2199)]),
        (6,  "Toys",   [(920,999),(3944,3945),(7993,7993),(7996,7996),(7999,7999)]),
        (7,  "Fun",    [(7800,7833),(7840,7841),(7900,7932),(7940,7949),
                        (7980,7983),(7990,7992),(7994,7995),(7997,7998)]),
        (8,  "Books",  [(2700,2749),(2770,2771),(2780,2799)]),
        (9,  "Hshld",  [(2047,2047),(2391,2392),(2510,2519),(2590,2599),(2840,2844),
                        (3160,3161),(3170,3172),(3190,3199),(3229,3229),(3260,3260),
                        (3262,3263),(3269,3269),(3630,3639),(3750,3751),(3800,3800),
                        (3860,3861),(3870,3873),(3910,3911),(3914,3915),(3960,3962),
                        (3991,3991),(3995,3995)]),
        (10, "Clths",  [(2300,2390),(3020,3021),(3100,3111),(3130,3131),
                        (3140,3149),(3150,3151),(3963,3965)]),
        (11, "Hlth",   [(8000,8099)]),
        (12, "MedEq",  [(3693,3693),(3841,3851),(5047,5047),(5122,5122),(5912,5912)]),
        (13, "Drugs",  [(2830,2836)]),
        (14, "Chems",  [(2810,2829),(2850,2869),(2870,2879),(2890,2899)]),
        (15, "Rubbr",  [(3031,3031),(3041,3041),(3050,3053),(3060,3099)]),
        (16, "Txtls",  [(2200,2269),(2290,2295),(2297,2299),(2393,2395),(2397,2399)]),
        (17, "BldMt",  [(800,899),(1500,1542),(1550,1560),(1600,1799),
                        (2400,2439),(2450,2459),(2490,2499),(2660,2661),(2950,2952),
                        (3200,3241),(3250,3275),(3280,3299),(3420,3442),(3446,3446),
                        (3448,3452),(3490,3499),(3996,3996)]),
        (18, "Steel",  [(3300,3399)]),
        (19, "FabPr",  [(3400,3419),(3443,3444),(3460,3469),(3462,3489)]),
        (20, "Mach",   [(3510,3536),(3538,3538),(3540,3569),(3580,3599)]),
        (21, "ElcEq",  [(3600,3629),(3640,3649),(3660,3660),(3690,3699),
                        (3810,3810),(3812,3812)]),
        (22, "Autos",  [(2296,2296),(2396,2396),(3010,3011),(3537,3537),(3647,3647),
                        (3694,3694),(3714,3716),(3792,3792),(5010,5015),
                        (5510,5521),(5530,5531),(5560,5571),(5580,5581),(5590,5599)]),
        (23, "Aero",   [(3720,3729)]),
        (24, "Ships",  [(3730,3732)]),
        (25, "Guns",   [(3480,3489),(3760,3769),(3795,3795)]),
        (26, "Gold",   [(1040,1049)]),
        (27, "Mines",  [(1000,1039),(1050,1099),(1400,1499)]),
        (28, "Coal",   [(1200,1299)]),
        (29, "Oil",    [(1300,1382),(1389,1389),(2911,2911),(2990,2999),(5171,5172)]),
        (30, "Util",   [(4900,4942)]),
        (31, "Telcm",  [(4800,4899)]),
        (32, "PerSv",  [(7020,7021),(7040,7049),(7200,7299),(7500,7501),
                        (7520,7549),(7600,7631),(7640,7641),(7690,7699)]),
        (33, "BusSv",  [(2750,2759),(3993,3993),(7372,7374),(7376,7384),
                        (7389,7399),(8700,8742),(8900,8999),(4220,4229)]),
        (34, "Comps",  [(3570,3579),(3680,3689),(3695,3695),(7370,7372),(7375,7375)]),
        (35, "Chips",  [(3622,3622),(3674,3674),(3675,3679),(3825,3825),(3827,3829)]),
        (36, "LabEq",  [(3559,3559),(3820,3826),(3829,3829),(3830,3839)]),
        (37, "Paper",  [(2520,2549),(2600,2661),(2670,2699),(2760,2761),(3950,3955)]),
        (38, "Boxes",  [(2650,2659)]),
        (39, "Trans",  [(4000,4013),(4040,4049),(4100,4173),(4190,4231),
                        (4400,4499),(4600,4699),(4700,4742),(4750,4757),(4780,4789)]),
        (40, "Whlsl",  [(5000,5099),(5110,5159),(5160,5199)]),
        (41, "Rtail",  [(5200,5999)]),
        (42, "Meals",  [(5800,5829),(5890,5899),(7000,7011),(7041,7041)]),
        (43, "Banks",  [(6000,6099),(6110,6113)]),
        (44, "Insur",  [(6300,6411)]),
        (45, "RlEst",  [(6500,6552),(6770,6770),(6798,6798)]),
        (46, "Fin",    [(6199,6299),(6400,6499),(6720,6749),(6790,6799)]),
        (47, "Other",  [(4950,4971),(4991,4991)]),
    ]
    for code, name, ranges in ff48_def:
        for lo, hi in ranges:
            fill(max(lo, 0), min(hi, 9999), code, name)
    return codes, names


_FF48_CODES, _FF48_NAMES = _build_ff48_lookup()


def sic_to_ff48(sic_series):
    """Vectorised SIC → (ff48_code, ff48_name).  Unrecognised SICs → (48, 'Other')."""
    sic_int = pd.to_numeric(sic_series, errors="coerce").fillna(0).astype(int).clip(0, 9999)
    codes = pd.Series(_FF48_CODES[sic_int.values], index=sic_series.index, dtype=int)
    names = pd.Series(_FF48_NAMES[sic_int.values], index=sic_series.index, dtype=str)
    return codes, names


# SIC priority: sich (historical, funda) → sic from comp.company (comp_company df).
# comp.funda has no `sic` column; comp.company.sic is the current SIC and is
# reliably populated for virtually all firms.
comp_sic_map = (comp_company.drop_duplicates(subset=["gvkey"])
                             .set_index("gvkey")["sic"])
comp_sic_fallback = pd.Series(
    comp_sic_map.reindex(funda_raw["gvkey"].to_numpy()).to_numpy(dtype=object),
    index=funda_raw.index,
)
sic_src = funda_raw["sich"].fillna(comp_sic_fallback)

ff48_codes, ff48_names = sic_to_ff48(sic_src)
funda_raw = funda_raw.copy()
funda_raw["ff48_industry"] = ff48_codes.values
funda_raw["ff48_name"]     = ff48_names.values

null_sic = sic_src.isna().sum()
print(f"SIC source coverage: sich={funda_raw['sich'].notna().sum():,}  "
      f"comp.company sic={comp_sic_fallback.notna().sum():,}  "
      f"no SIC at all={null_sic:,} (→ Other/48)")
print("\nFF48 industry distribution (top 15 by unique firms):")
dist = (funda_raw.groupby(["ff48_industry", "ff48_name"])["gvkey"]
        .nunique().sort_values(ascending=False))
print(dist.head(15).to_string())
print(f"\nOther/unclassified (ff48=48): "
      f"{(funda_raw['ff48_industry'] == 48).sum():,} rows  "
      f"({(funda_raw['ff48_industry'] == 48).mean():.1%})")

SIC source coverage: sich=12,547  comp.company sic=13,053  no SIC at all=0 (→ Other/48)

FF48 industry distribution (top 15 by unique firms):
ff48_industry  ff48_name
46             Fin          180
34             Comps        121
48             Other        109
43             Banks        103
13             Drugs         67
30             Util          66
17             BldMt         65
41             Rtail         65
44             Insur         63
33             BusSv         56
20             Mach          52
29             Oil           51
35             Chips         48
36             LabEq         46
12             MedEq         45

Other/unclassified (ff48=48): 882 rows  (6.8%)


In [23]:
# ── 8d. Cash Flow Ratios ────────────────────────────────────────────────────
# cash_flow_ratio    = (NI + D&A) / AT  — accrual-based earnings quality proxy
# operating_cf_ratio = oancf / AT       — cash-based measure (preferred)
# oancf is pulled via a small supplemental query; falls back to accrual proxy.

try:
    oancf_raw = conn.raw_sql(f"""
        SELECT gvkey, datadate, oancf
        FROM comp.funda
        WHERE gvkey IN ('{gvkey_sql}')
          AND fyear BETWEEN {YEAR_START} AND {YEAR_END}
          AND indfmt = 'INDL' AND datafmt = 'STD' AND popsrc = 'D' AND consol = 'C'
    """, date_cols=["datadate"])
    print(f"oancf rows: {len(oancf_raw):,}  |  "
          f"non-null: {oancf_raw['oancf'].notna().sum():,}")
except Exception as e:
    print(f"oancf pull failed ({type(e).__name__}) — using accrual proxy throughout")
    oancf_raw = pd.DataFrame(columns=["gvkey", "datadate", "oancf"])

# Base: funda_raw supplies ni, dp, at
cf_base = funda_raw[["gvkey", "datadate", "ni", "dp", "at"]].copy()
cf_base = cf_base.merge(
    oancf_raw[["gvkey", "datadate", "oancf"]], on=["gvkey", "datadate"], how="left"
)

_at = cf_base["at"]
cf_base["cash_flow_ratio"] = np.where(
    _at.notna() & (_at > 0.01),
    (cf_base["ni"].fillna(0) + cf_base["dp"].fillna(0)) / _at,
    np.nan
)
cf_base["operating_cf_ratio"] = np.where(
    _at.notna() & (_at > 0.01) & cf_base["oancf"].notna(),
    cf_base["oancf"] / _at,
    cf_base["cash_flow_ratio"]   # fallback to accrual proxy when oancf missing
)

cf_annual = cf_base[["gvkey", "datadate", "cash_flow_ratio", "operating_cf_ratio"]]

print(f"\ncash_flow_ratio    non-null: {cf_annual['cash_flow_ratio'].notna().sum():,}")
print(f"operating_cf_ratio non-null: {cf_annual['operating_cf_ratio'].notna().sum():,}")
print(cf_annual[["cash_flow_ratio", "operating_cf_ratio"]].describe().round(3))

oancf rows: 13,053  |  non-null: 12,942

cash_flow_ratio    non-null: 12,972
operating_cf_ratio non-null: 12,972
       cash_flow_ratio  operating_cf_ratio
count        12972.000           12972.000
mean             0.076               0.090
std              0.118               0.100
min             -2.550              -1.586
25%              0.028               0.043
50%              0.071               0.080
75%              0.118               0.129
max              1.704               1.804


In [24]:
# ── 8e. Business Segment Count (`comp.segw`) ─────────────────────────────────
# Number of distinct business segments proxies firm complexity / diversification.
# Requires the Compustat Segments subscription (same add-on as comp.segg).
# Falls back gracefully to NaN if unavailable.

SEG_W_AVAILABLE = False
SEG_W_TABLE     = None

for _tbl in ["comp.segw", "compa.segw"]:
    try:
        conn.raw_sql(f"SELECT 1 FROM {_tbl} LIMIT 1", date_cols=[])
        SEG_W_AVAILABLE = True
        SEG_W_TABLE     = _tbl
        print(f"✓  Business segments table: {_tbl}")
        break
    except Exception:
        print(f"✗  Not accessible: {_tbl}")

if SEG_W_AVAILABLE:
    bseg_raw = conn.raw_sql(f"""
        SELECT gvkey, datadate, COUNT(DISTINCT sid) AS n_business_segments
        FROM {SEG_W_TABLE}
        WHERE gvkey IN ('{gvkey_sql}')
          AND datadate BETWEEN '{YEAR_START}-01-01' AND '{YEAR_END}-12-31'
          AND stype = 'BUSSEG'
        GROUP BY gvkey, datadate
    """, date_cols=["datadate"])
    seg_annual = bseg_raw.copy()
    print(f"Business segment rows: {len(seg_annual):,}")
    print(seg_annual["n_business_segments"].describe().round(1))
else:
    seg_annual = pd.DataFrame(columns=["gvkey", "datadate", "n_business_segments"])
    print("n_business_segments will be NaN (no comp.segw access).")
    print("Request 'Compustat Segments' via your institution's WRDS portal.")

✗  Not accessible: comp.segw
✗  Not accessible: compa.segw
n_business_segments will be NaN (no comp.segw access).
Request 'Compustat Segments' via your institution's WRDS portal.


In [25]:
# ── 8f. VIX Annual Average (macro uncertainty control) ─────────────────────
# Annual average of the CBOE VIX index matched to Compustat fiscal year.
# For firms with non-December fiscal year ends, the VIX for the calendar year
# in which the fiscal year ends is used (standard in the literature).
# Values are hardcoded from CBOE historical data; update 2025 once final.

vix_data = {
    2012: 17.80, 2013: 14.23, 2014: 14.18, 2015: 16.67, 2016: 15.83,
    2017: 11.09, 2018: 16.64, 2019: 15.39, 2020: 29.25,
    2021: 19.66, 2022: 25.64, 2023: 16.86, 2024: 15.55,
    2025: 19.08,   # preliminary (Jan–Mar 2026 avg; update when full year available)
}

vix_annual = pd.DataFrame(
    [(yr, val) for yr, val in vix_data.items()],
    columns=["fyear", "vix_annual"]
)
vix_annual["fyear"] = vix_annual["fyear"].astype(int)

print("VIX annual averages (CBOE) for analysis window:")
window_mask = vix_annual["fyear"].between(YEAR_START, YEAR_END)
print(vix_annual[window_mask].to_string(index=False))

VIX annual averages (CBOE) for analysis window:
 fyear  vix_annual
  2017       11.09
  2018       16.64
  2019       15.39
  2020       29.25
  2021       19.66
  2022       25.64
  2023       16.86
  2024       15.55
  2025       19.08


## 9 Merge and derived variables

In [15]:
# ── 9a. Backbone: Compustat funda (includes ff48 from Section 8c) ───────────
panel = funda_raw.copy()

# Standardise CIK to integer (Compustat stores as string)
panel["cik"] = pd.to_numeric(panel["cik"].str.lstrip("0").replace("", "0"), errors="coerce").astype("Int64")

# ── 9b. Add CRSP market data ──────────────────────────────────────────────
panel = panel.merge(
    crsp_annual[["gvkey","datadate","permno","mktcap","ret_ann","beta"]],
    on=["gvkey","datadate"], how="left"
)

# ── 9c. Add geographic segments ───────────────────────────────────────────
panel = panel.merge(
    geo_annual[["gvkey","datadate","foreign_rev_share","n_geo_segments",
                "seg_total_rev","seg_foreign_rev"]],
    on=["gvkey","datadate"], how="left"
)

# ── 9d. Add institutional ownership ───────────────────────────────────────
panel = panel.merge(io_annual, on=["gvkey","datadate"], how="left")

# ── 9e. Add analyst coverage ──────────────────────────────────────────────
panel = panel.merge(ibes_annual, on=["gvkey","datadate"], how="left")

# ── 9f. Add Section 8 additional variables ───────────────────────────────
panel = panel.merge(sales_growth_annual, on=["gvkey","datadate"], how="left")
panel = panel.merge(firm_risk_annual,    on=["gvkey","datadate"], how="left")
panel = panel.merge(cf_annual,           on=["gvkey","datadate"], how="left")
panel = panel.merge(seg_annual,          on=["gvkey","datadate"], how="left")
# vix_annual merges on fyear (calendar year of fiscal year end)
panel["fyear"] = panel["fyear"].astype("Int64")
vix_annual["fyear"] = vix_annual["fyear"].astype("Int64")
panel = panel.merge(vix_annual, on="fyear", how="left")
# ff48_industry and ff48_name are already in funda_raw → panel

# ── 9g. Add ticker, index membership from universe via CIK ────────────────
panel = panel.merge(
    universe[["cik","Symbol","Index"]].rename(columns={"Symbol":"ticker","Index":"sp_index"}),
    on="cik", how="left"
)

print(f"Panel shape before derived variables: {panel.shape}")
print(f"Fiscal years: {sorted(panel['fyear'].dropna().astype(int).unique())}")
print(f"\nNew Section 8 variable coverage:")
for col in ["sales_growth","firm_risk","cash_flow_ratio","operating_cf_ratio",
            "n_business_segments","vix_annual","ff48_industry"]:
    pct = panel[col].notna().mean() * 100 if col in panel.columns else float("nan")
    print(f"  {col:30s}: {pct:.1f}%")

NameError: name 'funda_raw' is not defined

In [ ]:
# ── 9h. Derived financial variables ─────────────────────────────────────────

df = panel.copy()

# Coerce numeric columns that may have come through as object dtype from
# the row-wise CRSP apply() — np.log / arithmetic fail on Python float objects.
for _col in ["mktcap", "ret_ann", "beta", "mkvalt"]:
    if _col in df.columns:
        df[_col] = pd.to_numeric(df[_col], errors="coerce")

# Helpers
def safe_div(a, b, floor=None, cap=None):
    """Element-wise division with optional floor/cap."""
    r = np.where((b.notna()) & (b != 0), a / b, np.nan)
    r = pd.Series(r, index=a.index)
    if floor is not None: r = r.clip(lower=floor)
    if cap   is not None: r = r.clip(upper=cap)
    return r

at   = df["at"]       # total assets
sale = df["sale"]     # net revenue
ceq  = df["ceq"]     # common equity (book)

# ── Size ─────────────────────────────────────────────────────────────────────
df["log_at"]          = np.log(at.clip(lower=0.01))
df["log_sale"]        = np.log(sale.clip(lower=0.01))
df["log_mktcap"]      = np.log(df["mktcap"].clip(lower=0.01))   # from CRSP ($k)
df["mktcap_usd_bn"]   = df["mktcap"] / 1e6                       # convert $k → $bn

# ── Leverage ─────────────────────────────────────────────────────────────────
total_debt            = df["dltt"].fillna(0) + df["dlc"].fillna(0)
df["leverage"]        = safe_div(total_debt, at, floor=0, cap=5)
df["debt_to_equity"]  = safe_div(total_debt, ceq.clip(lower=0.01), floor=0, cap=20)
df["net_leverage"]    = safe_div(total_debt - df["che"].fillna(0), at, cap=5)

# ── Profitability ─────────────────────────────────────────────────────────────
df["roa"]             = safe_div(df["ni"],    at,   cap=1,  floor=-1)
df["roe"]             = safe_div(df["ni"],    ceq.clip(lower=0.01), cap=5, floor=-5)
df["ebitda_margin"]   = safe_div(df["oibdp"], sale, cap=1,  floor=-1)
df["gross_margin"]    = safe_div(sale - df["cogs"], sale, floor=-1, cap=1)
df["profit_margin"]   = safe_div(df["ni"],    sale, cap=1,  floor=-1)
df["asset_turnover"]  = safe_div(sale, at, floor=0, cap=5)

# ── Liquidity ─────────────────────────────────────────────────────────────────
df["cash_ratio"]      = safe_div(df["che"],  at,   floor=0, cap=1)
df["current_ratio"]   = safe_div(df["act"], df["lct"], floor=0, cap=10)

# ── Investment & Tangibility ──────────────────────────────────────────────────
df["capex_intensity"] = safe_div(df["capx"].fillna(0), at, floor=0, cap=1)
df["rd_intensity"]    = safe_div(df["xrd"].fillna(0),  at, floor=0, cap=1)
df["rd_missing"]      = df["xrd"].isna().astype(int)
df["tangibility"]     = safe_div(df["ppent"], at, floor=0, cap=1)
df["depreciation_rate"] = safe_div(df["dp"].fillna(0), df["ppent"].clip(lower=0.01), floor=0, cap=1)

# ── Market / Valuation ───────────────────────────────────────────────────────
mktcap_k = df["mktcap"].combine_first(df["mkvalt"].fillna(np.nan) * 1e3)
df["tobin_q"]         = safe_div(mktcap_k + df["lt"].fillna(0) * 1e3, at * 1e3, floor=0, cap=20)
df["book_to_market"]  = safe_div(ceq, mktcap_k / 1e3, floor=0, cap=10)

# ── International Exposure (from Income Statement) ────────────────────────────
df["foreign_income_share"] = np.where(
    (df["pi"].notna()) & (df["pi"] != 0) & (df["pifo"].notna()),
    (df["pifo"] / df["pi"]).clip(-2, 2),
    np.nan
)

# ── HHI-Index ────────────────────────────
df['rev_share'] = df.groupby(['ff48_industry', 'fiscal_year'])['sale'].transform(
    lambda x: x / x.sum()
)
df['hhi'] = df.groupby(['ff48_industry', 'fiscal_year'])['rev_share'].transform(
    lambda x: (x**2).sum()
)

# ── Litigation-Risk ────────────────────────────
HIGH_LITIG_SIC = list(range(2833, 2837)) + list(range(3570, 3577)) + \
                 list(range(3600, 3675)) + list(range(5200, 5961)) + \
                 list(range(7371, 7380))
df['high_litigation'] = df['sich'].isin(HIGH_LITIG_SIC).astype(int)

# ── Payout & Growth ──────────────────────────────────────────────────────────
df["dividend_yield"]  = safe_div(df["dvt"].fillna(0), mktcap_k / 1e3, floor=0, cap=1)
df["payout_ratio"]    = safe_div(df["dvt"].fillna(0), df["ni"].clip(lower=0.01), floor=0, cap=5)

# ── Analyst / Information ────────────────────────────────────────────────────
df["log_analyst_cov"] = np.log1p(df["analyst_coverage"].fillna(0))
df["eps_uncertainty"] = safe_div(df["eps_dispersion"].abs(), df["eps_median"].abs().clip(lower=0.01))

# ── Section 8 derived ────────────────────────────────────────────────────────
df["log_n_biz_seg"] = np.log1p(df["n_business_segments"].fillna(0))

# ── Flags ────────────────────────────────────────────────────────────────────
df["has_geo_seg"]     = df["foreign_rev_share"].notna().astype(int)
df["has_inst_own"]    = df["inst_ownership"].notna().astype(int)
df["has_ibes"]        = df["analyst_coverage"].notna().astype(int)

print("Derived variables computed.")
print(f"Panel shape: {df.shape}")
df[["log_at","leverage","roa","tobin_q","foreign_rev_share","inst_ownership",
    "analyst_coverage","ret_ann","beta","sales_growth","firm_risk",
    "cash_flow_ratio","vix_annual"]].describe().round(3)

Derived variables computed.
Panel shape: (13106, 93)


,log_at,leverage,roa,tobin_q,foreign_rev_share,analyst_coverage,ret_ann,beta,sales_growth,firm_risk,cash_flow_ratio,vix_annual
count,13025.0,13025.000,13022.000,12751.000,8161.000,12582.0,11342.000,12523.000,11508.000,10029.000,13025.000,13106.000
mean,8.829,0.311,0.046,2.325,0.392,11.371,0.172,1.189,0.120,0.044,0.076,18.844
std,1.634,0.245,0.102,2.145,0.566,7.977,0.931,0.655,0.363,0.072,0.118,5.248
min,1.648,0.000,-1.000,0.382,-1.000,1.0,-0.980,-3.412,-1.000,0.000,-2.550,11.090
25%,7.697,0.121,0.012,1.146,0.033,5.0,-0.096,0.794,-0.003,0.011,0.028,15.550
50%,8.728,0.293,0.041,1.608,0.269,9.0,0.111,1.130,0.071,0.025,0.071,16.860
75%,9.835,0.437,0.083,2.592,0.703,16.0,0.337,1.507,0.166,0.049,0.118,19.660
max,15.303,3.892,1.000,20.000,2.000,62.0,83.635,7.816,5.000,1.178,1.704,29.250


In [29]:
# ── 9i. Winsorise continuous variables at 1%/99% ────────────────────────────
# Avoids regression results driven by data errors or extreme outliers.
# Raw Compustat values are kept; only the winsorised columns are added.

WINSORISE_COLS = [
    "leverage", "net_leverage", "debt_to_equity",
    "roa", "roe", "ebitda_margin", "gross_margin", "profit_margin", "asset_turnover",
    "cash_ratio", "current_ratio",
    "capex_intensity", "rd_intensity", "tangibility",
    "tobin_q", "book_to_market",
    "foreign_income_share", "foreign_rev_share",
    "inst_ownership", "ret_ann", "beta",
    "dividend_yield", "payout_ratio",
    # Section 8 additions
    "sales_growth", "firm_risk", "cash_flow_ratio", "operating_cf_ratio",
]

for col in WINSORISE_COLS:
    if col in df.columns:
        lo = df[col].quantile(0.01)
        hi = df[col].quantile(0.99)
        df[f"{col}_w"] = df[col].clip(lo, hi)

print(f"Winsorised columns added: {[c+'_w' for c in WINSORISE_COLS if c in df.columns][:5]} ...")
print(f"Final panel shape: {df.shape}")

Winsorised columns added: ['leverage_w', 'net_leverage_w', 'debt_to_equity_w', 'roa_w', 'roe_w'] ...
Final panel shape: (13106, 119)


## 10 Save output

In [30]:
# ── Column ordering ──────────────────────────────────────────────────────────
ID_COLS = ["gvkey", "cik", "ticker", "conm", "sp_index", "fic", "sic", "sich",
           "naics", "ff48_industry", "ff48_name",
           "fyear", "fyr", "datadate", "permno"]

MARKET_COLS = ["mktcap_usd_bn", "log_mktcap", "ret_ann", "beta",
               "ret_ann_w", "beta_w"]

ACCOUNTING_COLS = [
    "at", "sale", "revt", "ni", "ib", "oibdp", "oiadp",
    "dltt", "dlc", "lt", "ceq", "seq",
    "act", "lct", "che", "ppent", "capx", "xrd", "dp",
    "csho", "prcc_f", "mkvalt", "dvt",
    "pifo", "pi",
]

DERIVED_COLS = [
    "log_at", "log_sale",
    "leverage", "leverage_w", "net_leverage", "net_leverage_w", "debt_to_equity", "debt_to_equity_w",
    "roa", "roa_w", "roe", "roe_w",
    "ebitda_margin", "ebitda_margin_w", "gross_margin", "gross_margin_w", "profit_margin", "profit_margin_w", "asset_turnover",
    "cash_ratio", "cash_ratio_w", "current_ratio", "current_ratio_w",
    "capex_intensity", "capex_intensity_w",
    "rd_intensity", "rd_intensity_w", "rd_missing",
    "tangibility", "tangibility_w", "depreciation_rate",
    "tobin_q", "tobin_q_w", "book_to_market", "book_to_market_w",
    "foreign_income_share", "foreign_income_share_w",
    "dividend_yield", "dividend_yield_w", "payout_ratio", "payout_ratio_w",
    # Section 8 additions
    "sales_growth", "sales_growth_w",
    "firm_risk", "firm_risk_w",
    "cash_flow_ratio", "cash_flow_ratio_w",
    "operating_cf_ratio", "operating_cf_ratio_w",
    "n_business_segments", "log_n_biz_seg",
]

GEO_COLS = ["foreign_rev_share", "foreign_rev_share_w", "n_geo_segments",
            "seg_total_rev", "seg_foreign_rev", "has_geo_seg"]

IO_COLS  = ["inst_ownership", "inst_ownership_w", "has_inst_own"]

IBES_COLS = ["analyst_coverage", "log_analyst_cov", "eps_median",
             "eps_dispersion", "eps_uncertainty", "has_ibes"]

MACRO_COLS = ["vix_annual"]

all_cols = (ID_COLS + MARKET_COLS + ACCOUNTING_COLS + DERIVED_COLS
            + GEO_COLS + IO_COLS + IBES_COLS + MACRO_COLS)
final_cols = [c for c in all_cols if c in df.columns]

output = df[final_cols].sort_values(["gvkey","fyear"]).reset_index(drop=True)

before = len(output)
output = output.drop_duplicates(subset=["gvkey","fyear"], keep="first")
print(f"Rows before dedup: {before:,}  →  after: {len(output):,}")

print(f"\nFinal panel: {output['gvkey'].nunique():,} firms × {output['fyear'].nunique()} fiscal years")
print(f"Columns: {len(output.columns)}")

print("\nVariable coverage (% non-null) — core controls:")
cov_cols = ["log_at","leverage","roa","gross_margin","tobin_q","log_mktcap","ret_ann","beta",
            "foreign_rev_share","inst_ownership","analyst_coverage",
            "sales_growth","firm_risk","cash_flow_ratio","operating_cf_ratio",
            "n_business_segments","vix_annual","ff48_industry"]
print((output[[c for c in cov_cols if c in output.columns]].notna().mean() * 100).round(1).to_string())

out_path = DATA_OUT / f"financials_{OUT_SUFFIX}.parquet"
output.to_parquet(out_path, index=False, compression="zstd")
print(f"\nSaved → {out_path}  ({out_path.stat().st_size / 1e6:.1f} MB)")

Rows before dedup: 13,106  →  after: 13,052

Final panel: 1,491 firms × 9 fiscal years
Columns: 109

Variable coverage (% non-null) — core controls:
log_at                  99.4
leverage                99.4
roa                     99.4
tobin_q                 97.3
log_mktcap              85.8
ret_ann                 86.5
beta                    95.6
foreign_rev_share       62.2
inst_ownership           0.0
analyst_coverage        96.0
sales_growth            87.8
firm_risk               76.5
cash_flow_ratio         99.4
operating_cf_ratio      99.4
n_business_segments      0.0
vix_annual             100.0
ff48_industry          100.0

Saved → c:\Users\Sören Schlißke\Dateien\Uni\MACFin\Thesis\Python\data\processed\financials_2017_2025.parquet  (7.6 MB)


In [31]:
# ── Quick sanity checks ──────────────────────────────────────────────────────
print("=== Sanity Checks ===\n")

# 1. Apple as a spot check (CIK 320193)
apple = output[output["cik"] == 320193][
    ["ticker","fyear","log_at","leverage","roa","tobin_q","foreign_rev_share",
     "inst_ownership","ff48_industry","ff48_name","vix_annual"]
]
print("Apple Inc (CIK 320193):"); print(apple.tail(5).to_string(index=False)); print()

# 2. Check no duplicates
dup = output.duplicated(subset=["gvkey","fyear"]).sum()
print(f"Duplicate gvkey×fyear rows: {dup}  (should be 0)")

# 3. Fiscal year coverage by year
print("\nFirms per fiscal year:")
print(output.groupby("fyear")["gvkey"].nunique().to_string())

# 4. Missing rate for key regression variables
print("\nMissing (%) for key regression controls:")
reg_controls = [
    "log_at","leverage_w","roa_w","tobin_q_w","cash_ratio_w",
    "rd_intensity_w","tangibility_w","ret_ann_w","beta_w",
    "foreign_rev_share_w","inst_ownership_w","log_analyst_cov",
    "sales_growth_w","firm_risk_w","cash_flow_ratio_w","operating_cf_ratio_w",
    "n_business_segments","vix_annual","ff48_industry",
]
missing_pct = output[[c for c in reg_controls if c in output.columns]].isna().mean().mul(100).round(1)
print(missing_pct.to_string())

# 5. FF48 industry spot check
print("\nFF48 distribution (top 10):")
print(output.groupby(["ff48_industry","ff48_name"])["gvkey"]
      .nunique().sort_values(ascending=False).head(10).to_string())

=== Sanity Checks ===

Apple Inc (CIK 320193):
ticker  fyear     log_at  leverage      roa   tobin_q  foreign_rev_share inst_ownership  ff48_industry ff48_name  vix_annual
  AAPL   2021  12.768547  0.388949 0.269742  7.442414           0.629081            NaN             48     Other       19.66
  AAPL   2022  12.773529  0.375558 0.282924  7.102562           0.598642            NaN             48     Other       25.64
  AAPL   2023  12.773041  0.351492 0.275098  8.374661           0.640958            NaN             48     Other       16.86
  AAPL   2024  12.807598  0.326207 0.256825 10.494386           0.625987            NaN             48     Other       15.55
  AAPL   2025  12.791749  0.312818 0.311796 11.266039           0.617800            NaN             48     Other       19.08

Duplicate gvkey×fyear rows: 0  (should be 0)

Firms per fiscal year:
fyear
2017    1403
2018    1429
2019    1453
2020    1466
2021    1480
2022    1490
2023    1491
2024    1491
2025    1349

Missing (

## 11 Analysis panel: join with the NLP features

NLP fiscal years are converted to the Compustat convention before merging; controls are lagged one year.

In [2]:
from pathlib import Path
import numpy as np, pandas as pd
ROOT      = Path.cwd().parent              # project root
DATA_OUT  = ROOT / "data" / "processed"    # where 06 reads the panel
TEST_MODE = False
output = pd.read_parquet(DATA_OUT / "financials_2017_2025.parquet")

# Optional H2 proxies: coded in cell 35 but never saved
output["rev_share"] = output.groupby(["ff48_industry", "fyear"])["sale"].transform(lambda x: x / x.sum())
output["hhi"] = output.groupby(["ff48_industry", "fyear"])["rev_share"].transform(lambda x: (x ** 2).sum())
HIGH_LITIG_SIC = (list(range(2833, 2837)) + list(range(3570, 3577)) + list(range(3600, 3675))
                  + list(range(5200, 5961)) + list(range(7371, 7380)))
output["high_litigation"] = output["sich"].isin(HIGH_LITIG_SIC).astype(int)

In [5]:
import wrds, shutil
conn = wrds.Connection(wrds_username='soerenssch')

def first_table(library, candidates):
    """Return the first table in `candidates` that exists in `library`."""
    tabs = set(conn.list_tables(library=library))
    for t in candidates:
        if t in tabs:
            return t
    raise ValueError(f'None of {candidates} exists in {library}')

def pick_cols(library, table, wanted):
    """Map aliases to the first matching column name, so the SQL survives naming differences."""
    avail = set(conn.describe_table(library, table)['name'].str.lower())
    got = {k: next((c for c in cands if c in avail), None) for k, cands in wanted.items()}
    print(f'{library}.{table}: {got}')
    return got

def as_float(s):
    """WRDS returns nullable columns (pd.NA); convert to plain float64 with NaN."""
    return pd.Series(pd.to_numeric(s, errors='coerce').to_numpy(dtype='float64', na_value=np.nan),
                     index=s.index)

WRDS recommends setting up a .pgpass file.
pgpass file created at C:\Users\Sören Schlißke\AppData\Roaming\postgresql\pgpass.conf
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


In [6]:
# CRSP monthly from CIZ; recompute mktcap, ret_ann, beta ────────
# Same rules as Section 4: 12-month buy-and-hold return (>= 6 months), 36-month
# beta on monthly returns (>= 12 months). Guard: NaN if the fiscal-year-end
# month lies beyond CRSP's last available month (no stale FY2025 betas).
BETA_WINDOW = 36
CIZ_START, CIZ_END = '2014-12-01', '2026-12-31'

m_tab = first_table('crsp', ['msf_v2', 'stkmthsecuritydata'])
mc = pick_cols('crsp', m_tab, {'date': ['mthcaldt'], 'ret': ['mthret'],
                               'prc': ['mthprc'], 'shrout': ['shrout', 'mthshrout']})
assert all(mc.values()), 'A column is missing - check the printed column map'

permnos = output['permno'].dropna().astype(int).unique().tolist()
chunks = []
for i in range(0, len(permnos), 500):
    p = ','.join(map(str, permnos[i:i + 500]))
    chunks.append(conn.raw_sql(f"""
        SELECT permno, {mc['date']} AS date, {mc['ret']} AS ret,
               ABS({mc['prc']}) AS prc, {mc['shrout']} AS shrout
        FROM crsp.{m_tab}
        WHERE permno IN ({p}) AND {mc['date']} BETWEEN '{CIZ_START}' AND '{CIZ_END}'
    """, date_cols=['date']))
msf = pd.concat(chunks, ignore_index=True)
msf['ym'] = msf['date'].dt.to_period('M')
msf = msf.sort_values(['permno', 'date']).drop_duplicates(['permno', 'ym'], keep='last')
for c in ['ret', 'prc', 'shrout']:
       msf[c] = as_float(msf[c])
msf['mktcap'] = msf['prc'] * msf['shrout']

ic = pick_cols('crsp', 'indmthseriesdata', {'date': ['mthcaldt'], 'ret': ['mthtotret']})
mkt = conn.raw_sql(f"""
    SELECT {ic['date']} AS date, {ic['ret']} AS mkt_ret
    FROM crsp.indmthseriesdata
    WHERE indno = 1000200 AND {ic['date']} BETWEEN '{CIZ_START}' AND '{CIZ_END}'
""", date_cols=['date'])                      # 1000200 = CRSP value-weighted index
mkt['ym'] = mkt['date'].dt.to_period('M')
mkt = mkt.drop_duplicates('ym')[['ym', 'mkt_ret']]
mkt['mkt_ret'] = as_float(mkt['mkt_ret'])

LAST_YM = min(msf['ym'].max(), mkt['ym'].max())
print(f'CRSP monthly: {len(msf):,} rows | {msf.permno.nunique():,} permnos | last month {LAST_YM}')

m = msf.merge(mkt, on='ym', how='left')
by_p = {p: d.set_index('ym').sort_index() for p, d in m.groupby('permno')}

def crsp_vars(permno, datadate):
    if pd.isna(permno) or pd.isna(datadate):
        return (np.nan, np.nan, np.nan)
    d = by_p.get(int(permno))
    e = pd.Period(pd.Timestamp(datadate), freq='M')
    if d is None or e > LAST_YM:
        return (np.nan, np.nan, np.nan)
    mcap = d['mktcap'].get(e, np.nan)
    r = d.loc[e - 11:e, 'ret'].dropna()
    ret_ann = (1 + r).prod() - 1 if len(r) >= 6 else np.nan
    b = d.loc[e - (BETA_WINDOW - 1):e, ['ret', 'mkt_ret']].dropna()
    beta = np.polyfit(b['mkt_ret'], b['ret'], 1)[0] if len(b) >= 12 else np.nan
    return (mcap, ret_ann, beta)

new = pd.DataFrame([crsp_vars(p, d) for p, d in zip(output['permno'], output['datadate'])],
                   columns=['mktcap', 'ret_ann', 'beta'], index=output.index).astype(float)

# Check against the legacy values for FY2018-2023 (should agree closely)
old_mcap = output['mktcap_usd_bn'] * 1e6
chk = output['fyear'].between(2018, 2023)
ov = chk & new['mktcap'].notna() & old_mcap.notna()
ratio = (new.loc[ov, 'mktcap'] / old_mcap[ov]).median()
if 900 < ratio < 1100:                     # shrout reported in shares, not thousands
    new['mktcap'] /= 1000
    ratio = (new.loc[ov, 'mktcap'] / old_mcap[ov]).median()
print(f'median mktcap ratio new/old (FY2018-23): {ratio:.4f}  (expect ~1)')
for c in ['ret_ann', 'beta']:
    ok = chk & new[c].notna() & output[c].notna()
    print(f'{c}: corr new vs old (FY2018-23) = {new.loc[ok, c].corr(output.loc[ok, c]):.4f}  N={ok.sum():,}')
print('\nShare non-missing by fyear (new):')
print(new.assign(fyear=output['fyear']).groupby('fyear')[['mktcap', 'ret_ann', 'beta']]
      .agg(lambda x: x.notna().mean()).round(3).to_string())

# Replace market variables and everything derived from market cap
shutil.copy2(DATA_OUT / 'financials_2017_2025.parquet',
             DATA_OUT / 'financials_2017_2025_SIZ_backup.parquet')
output['ret_ann']       = new['ret_ann']
output['beta']          = new['beta']
output['mktcap_usd_bn'] = new['mktcap'] / 1e6
output['log_mktcap']    = np.log(new['mktcap'].clip(lower=0.01))

def _ratio(a, b, floor, cap):
    r = pd.Series(np.where(b.notna() & (b != 0), a / b, np.nan), index=a.index)
    return r.clip(lower=floor, upper=cap)

mktcap_k = new['mktcap'].combine_first(output['mkvalt'] * 1e3)       # same fallback as cell 35
output['tobin_q']        = _ratio(mktcap_k + output['lt'].fillna(0) * 1e3, output['at'] * 1e3, 0, 20)
output['book_to_market'] = _ratio(output['ceq'], mktcap_k / 1e3, 0, 10)
for c in ['ret_ann', 'beta', 'tobin_q', 'book_to_market']:
    lo, hi = output[c].quantile([0.01, 0.99])
    output[f'{c}_w'] = output[c].clip(lo, hi)

output.to_parquet(DATA_OUT / 'financials_2017_2025.parquet', index=False, compression='zstd')
print('financials_2017_2025.parquet updated (backup: financials_2017_2025_SIZ_backup.parquet)')

Approximately 5240430 rows in crsp.msf_v2.
crsp.msf_v2: {'date': 'mthcaldt', 'ret': 'mthret', 'prc': 'mthprc', 'shrout': 'shrout'}
Approximately 4240 rows in crsp.indmthseriesdata.
crsp.indmthseriesdata: {'date': 'mthcaldt', 'ret': 'mthtotret'}
CRSP monthly: 185,791 rows | 1,512 permnos | last month 2025-12
median mktcap ratio new/old (FY2018-23): 1.0000  (expect ~1)
ret_ann: corr new vs old (FY2018-23) = 0.9998  N=8,454
beta: corr new vs old (FY2018-23) = 0.9986  N=8,381

Share non-missing by fyear (new):
       mktcap  ret_ann   beta
fyear                        
2017    0.944    0.937  0.924
2018    0.952    0.943  0.933
2019    0.951    0.946  0.937
2020    0.960    0.950  0.941
2021    0.978    0.973  0.955
2022    0.977    0.973  0.970
2023    0.986    0.983  0.976
2024    0.995    0.992  0.987
2025    0.974    0.971  0.970
financials_2017_2025.parquet updated (backup: financials_2017_2025_SIZ_backup.parquet)


In [7]:
# ── 11a. Load and deduplicate NLP features ──────────────────────────────────
nlp = pd.read_parquet(ROOT / "data" / "features" / "nlp_features.parquet")

# Keep only 2018–2025 (the analysis window)
nlp = nlp[nlp["fiscal_year"].between(2018, 2025)].copy()

# Deduplicate: if a firm filed an amended 10-K/A, keep the latest version
before_dedup = len(nlp)
nlp = (nlp
    .sort_values("filing_date", ascending=False)
    .drop_duplicates(subset=["cik", "fiscal_year"], keep="first")
    .reset_index(drop=True)
)
print(f"NLP rows: {before_dedup:,}  →  after dedup: {len(nlp):,}")
print(f"Unique firms: {nlp['cik'].nunique():,}  |  Years: {sorted(nlp['fiscal_year'].unique())}")
print(f"\nFirms per year:")
print(nlp.groupby("fiscal_year")["cik"].nunique().to_string())

NLP rows: 10,962  →  after dedup: 10,962
Unique firms: 1,488  |  Years: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

Firms per year:
fiscal_year
2018    1183
2019    1272
2020    1320
2021    1399
2022    1424
2023    1437
2024    1465
2025    1462


In [8]:
# Compustat convention: year-ends in Jan-May belong to the previous fyear
rd = pd.to_datetime(nlp["report_date"])
nlp["fyear_cs"] = pd.Series(np.where(rd.dt.month <= 5, rd.dt.year - 1, rd.dt.year),
                            index=nlp.index).astype("Int64")

In [9]:
# ── 11b. Merge: LEFT join from NLP onto financials ───────────────────────────
# Join key: cik (integer) + fiscal_year
fin = output.rename(columns={"fyear": "fiscal_year"}).copy()
fin["cik"] = fin["cik"].astype("Int64")
nlp["cik"] = nlp["cik"].astype("Int64")

analysis = nlp.merge(
    fin,
    on=["cik", "fiscal_year"],
    how="left",
    suffixes=("", "_fin")
)

print(f"Analysis panel: {len(analysis):,} rows  |  {analysis['cik'].nunique():,} unique firms")
print(f"  Matched to Compustat: {analysis['gvkey'].notna().sum():,} ({analysis['gvkey'].notna().mean():.1%})")
print(f"  Unmatched (no Compustat): {analysis['gvkey'].isna().sum():,}")

# ── 11c. Add 1-year lagged financial controls ────────────────────────────────
# Lag key controls by 1 year (standard in geo-risk regressions).
# vix_annual is a macro variable — include as contemporaneous (same year).
LAG_COLS = [
    "log_at", "leverage_w", "roa_w", "tobin_q_w", "cash_ratio_w",
    "rd_intensity_w", "tangibility_w", "foreign_rev_share_w",
    "inst_ownership_w", "log_analyst_cov", "ret_ann_w", "beta_w",
    "log_mktcap", "book_to_market_w",
    # Section 8 additions
    "sales_growth_w", "firm_risk_w", "cash_flow_ratio_w", "operating_cf_ratio_w",
]

lag_src = fin[["cik", "fiscal_year"] + [c for c in LAG_COLS if c in fin.columns]].copy()
# Shift: lag_year = t-1, so join on fiscal_year = t against fin fiscal_year = t-1
lag_src["fiscal_year"] = lag_src["fiscal_year"] + 1
lag_src = lag_src.rename(columns={c: f"L_{c}" for c in LAG_COLS if c in lag_src.columns})

analysis = analysis.merge(lag_src, on=["cik", "fiscal_year"], how="left")

lag_col_names = [f"L_{c}" for c in LAG_COLS if c in fin.columns]
lag_coverage = analysis[lag_col_names].notna().mean().mul(100).round(1)
print(f"\nLagged control coverage (% non-null):")
print(lag_coverage.to_string())

Analysis panel: 10,962 rows  |  1,488 unique firms
  Matched to Compustat: 10,820 (98.7%)
  Unmatched (no Compustat): 142

Lagged control coverage (% non-null):
L_log_at                  99.7
L_leverage_w              99.7
L_roa_w                   99.7
L_tobin_q_w               98.4
L_cash_ratio_w            99.7
L_rd_intensity_w          99.7
L_tangibility_w           94.1
L_foreign_rev_share_w     61.8
L_inst_ownership_w         0.0
L_log_analyst_cov         99.9
L_ret_ann_w               97.7
L_beta_w                  96.7
L_log_mktcap              98.2
L_book_to_market_w        98.4
L_sales_growth_w          88.6
L_firm_risk_w             76.4
L_cash_flow_ratio_w       99.7
L_operating_cf_ratio_w    99.7


In [10]:
fin = output.rename(columns={"fyear": "fyear_cs"}).copy()
fin["cik"] = fin["cik"].astype("Int64");  nlp["cik"] = nlp["cik"].astype("Int64")
fin["fyear_cs"] = fin["fyear_cs"].astype("Int64")
analysis = nlp.merge(fin, on=["cik", "fyear_cs"], how="left", suffixes=("", "_fin"))

In [11]:
lag_src = fin[["cik", "fyear_cs"] + [c for c in LAG_COLS if c in fin.columns]].copy()
lag_src["fyear_cs"] = lag_src["fyear_cs"] + 1
lag_src = lag_src.rename(columns={c: f"L_{c}" for c in LAG_COLS if c in lag_src.columns})
analysis = analysis.merge(lag_src, on=["cik", "fyear_cs"], how="left")

In [12]:
dd = pd.to_datetime(analysis["datadate"]); rd = pd.to_datetime(analysis["report_date"])
ok = ((dd - rd).abs() <= pd.Timedelta(days=14))[dd.notna()]
print("Compustat datadate within 14 days of 10-K report date:", round(ok.mean(), 4))

Compustat datadate within 14 days of 10-K report date: 0.9991


In [13]:
# ── 11d. Final panel summary & save ─────────────────────────────────────────
print("=== Analysis Panel Summary ===\n")
print(f"Rows: {len(analysis):,}  |  Firms: {analysis['cik'].nunique():,}  |  "
      f"Years: {sorted(analysis['fiscal_year'].unique())}")

print("\nFirms per year (unbalanced panel):")
print(analysis.groupby("fiscal_year")["cik"].nunique().to_string())

print("\nCoverage of lagged controls (% non-null):")
print(analysis[[f"L_{c}" for c in LAG_COLS if f"L_{c}" in analysis.columns]]
      .notna().mean().mul(100).round(1).to_string())

analysis_suffix = "test_2018" if TEST_MODE else "2018_2025"
analysis_path = DATA_OUT / f"analysis_panel_{analysis_suffix}.parquet"
analysis.to_parquet(analysis_path, index=False, compression="zstd")
print(f"\nSaved → {analysis_path}  ({analysis_path.stat().st_size / 1e6:.1f} MB)")
print("Ready for regression analysis.")

=== Analysis Panel Summary ===

Rows: 10,962  |  Firms: 1,488  |  Years: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

Firms per year (unbalanced panel):
fiscal_year
2018    1183
2019    1272
2020    1320
2021    1399
2022    1424
2023    1437
2024    1465
2025    1462

Coverage of lagged controls (% non-null):
L_log_at                  98.7
L_leverage_w              98.7
L_roa_w                   98.7
L_tobin_q_w               97.3
L_cash_ratio_w            98.7
L_rd_intensity_w          98.7
L_tangibility_w           93.1
L_foreign_rev_share_w     61.1
L_inst_ownership_w         0.0
L_log_analyst_cov         98.9
L_ret_ann_w               96.5
L_beta_w                  95.6
L_log_mktcap              97.1
L_book_to_market_w        97.3
L_sales_growth_w          87.5
L_firm_risk_w             75.1
L_cash_flow_ratio_w       98.7
L_operating_cf_ratio_w    98.7

Saved → c:\Users\Sören Schlißke\Dateien\Uni\

## 12 CRSP daily returns around 10-K filing dates

Day 0 is the first trading day on or after the filing date.

In [16]:
# ── 12a.i  Event-window configuration ────────────────────────────────────────

# Trading-day window around the filing date
EVENT_PRE_DAYS   = 260   # trading days before filing date (covers estimation window + buffer)
EVENT_POST_DAYS  = 30    # trading days after filing date

# Calendar-day query buffer sent to WRDS (larger than trading-day window to account
# for weekends/holidays; 400 cal days ≈ 278 trading days > 260 needed pre-event)
CAL_BUFFER_PRE   = 400
CAL_BUFFER_POST  = 50

# Query batch size: number of PERMNOs per SQL chunk (keep ≤ 500 to avoid WRDS timeouts)
DSF_BATCH_SIZE   = 300

# Output path for the final event-window daily return panel
CRSP_DSF_OUTPUT  = DATA_INTER / "crsp_daily_returns_around_filing.parquet"

print("── Section 12: CRSP Daily Event-Window Returns ──")
print(f"  Pre-event window   : {EVENT_PRE_DAYS} trading days")
print(f"  Post-event window  : {EVENT_POST_DAYS} trading days")
print(f"  Calendar query buf : −{CAL_BUFFER_PRE} / +{CAL_BUFFER_POST} days")
print(f"  WRDS batch size    : {DSF_BATCH_SIZE} permnos")
print(f"  Output             : {CRSP_DSF_OUTPUT}")

── Section 12: CRSP Daily Event-Window Returns ──
  Pre-event window   : 260 trading days
  Post-event window  : 30 trading days
  Calendar query buf : −400 / +50 days
  WRDS batch size    : 300 permnos
  Output             : ..\data\intermediate\crsp_daily_returns_around_filing.parquet


In [17]:
# ── 12a.ii  Build firm-year event panel (permno × filing_date) ───────────────
#
# Source A: NLP features parquet  →  cik, fiscal_year, filing_date
# Source B: Identifier bridge     →  cik → gvkey + permno  (with link validity dates)
#
# Link-validity rule: retain a bridge link for a filing event only when
#     linkdt  ≤ filing_date  ≤ linkenddt   (NaT means "open", i.e. no bound)
# If multiple valid links survive for a firm-year, keep one deterministically
# by sorting on linkdt (most recent start date first).

# ── Load filing dates ────────────────────────────────────────────────────────
nlp_filing = pd.read_parquet(ROOT /"data" / "features" / "nlp_features.parquet",
    columns=["cik", "fiscal_year", "filing_date"],
)
nlp_filing = (
    nlp_filing
    .dropna(subset=["filing_date"])
    .drop_duplicates(subset=["cik", "fiscal_year"])
    .copy()
)
nlp_filing["cik"]         = pd.to_numeric(nlp_filing["cik"], errors="coerce").astype("Int64")
nlp_filing["filing_date"] = pd.to_datetime(nlp_filing["filing_date"])
n_events_total = len(nlp_filing)

print(f"Firm-year filing events from NLP features : {n_events_total:,}")

# ── Load identifier bridge ───────────────────────────────────────────────────
bridge_df = pd.read_parquet(DATA_INTER / "identifier_bridge.parquet")
bridge_df = bridge_df[["cik", "gvkey", "permno", "linkdt", "linkenddt"]].copy()
bridge_df["cik"]       = pd.to_numeric(bridge_df["cik"],    errors="coerce").astype("Int64")
bridge_df["permno"]    = pd.to_numeric(bridge_df["permno"], errors="coerce").astype("Int64")
bridge_df["linkdt"]    = pd.to_datetime(bridge_df["linkdt"],    errors="coerce")
bridge_df["linkenddt"] = pd.to_datetime(bridge_df["linkenddt"], errors="coerce")
bridge_df = bridge_df.dropna(subset=["cik", "permno"])

# ── Merge & apply link-validity filter ──────────────────────────────────────
events_all = nlp_filing.merge(bridge_df, on="cik", how="left")

valid_link = (
    events_all["permno"].notna()
    & (events_all["linkdt"].isna()    | (events_all["filing_date"] >= events_all["linkdt"]))
    & (events_all["linkenddt"].isna() | (events_all["filing_date"] <= events_all["linkenddt"]))
)
events_all = events_all[valid_link].copy()

# Keep one link per firm-year (most recent linkdt wins for determinism)
events_all = (
    events_all
    .sort_values(["cik", "fiscal_year", "linkdt"], ascending=[True, True, False],
                 na_position="last")
    .drop_duplicates(subset=["cik", "fiscal_year"])
    .drop(columns=["linkdt", "linkenddt"])
    .reset_index(drop=True)
)

events_all["permno"] = events_all["permno"].astype(int)
events_all["gvkey"]  = events_all["gvkey"].astype(str)

# ── Coverage log ─────────────────────────────────────────────────────────────
n_with_permno  = len(events_all)
n_no_permno    = n_events_total - n_with_permno

print(f"  With valid permno (link-validated)   : {n_with_permno:,}")
print(f"  Dropped — no valid permno or link    : {n_no_permno:,}")
print(f"\nUnique permnos  : {events_all['permno'].nunique():,}")
print(f"Fiscal years    : {sorted(events_all['fiscal_year'].unique().tolist())}")
print(f"Filing date range: {events_all['filing_date'].min().date()} → "
      f"{events_all['filing_date'].max().date()}")

Firm-year filing events from NLP features : 10,962
  With valid permno (link-validated)   : 10,915
  Dropped — no valid permno or link    : 47

Unique permnos  : 1,499
Fiscal years    : [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Filing date range: 2018-02-26 → 2026-03-13


In [18]:
# ── 12b. Pull CRSP daily stock returns (CIZ) ──────────────────────────────────
d_tab = first_table('crsp', ['dsf_v2', 'stkdlysecuritydata'])
dc = pick_cols('crsp', d_tab, {'date': ['dlycaldt'], 'ret': ['dlyret'], 'prc': ['dlyprc'],
                               'vol': ['dlyvol'], 'shrout': ['shrout', 'dlyshrout']})
assert dc['date'] and dc['ret'] and dc['prc'] and dc['vol'], 'Column missing - check map'
shr = dc['shrout'] or 'NULL'

all_permnos  = sorted(events_all['permno'].unique().tolist())
global_start = (events_all['filing_date'].min() - pd.Timedelta(days=CAL_BUFFER_PRE)).strftime('%Y-%m-%d')
global_end   = (events_all['filing_date'].max() + pd.Timedelta(days=CAL_BUFFER_POST)).strftime('%Y-%m-%d')
print(f'Pulling crsp.{d_tab} | {len(all_permnos):,} permnos | {global_start} -> {global_end}')

dsf_chunks = []
for i in range(0, len(all_permnos), DSF_BATCH_SIZE):
    perm_str = ', '.join(str(p) for p in all_permnos[i:i + DSF_BATCH_SIZE])
    dsf_chunks.append(conn.raw_sql(f"""
        SELECT permno, {dc['date']} AS date, {dc['ret']} AS ret, ABS({dc['prc']}) AS prc,
               {dc['vol']} AS vol, {shr} AS shrout
        FROM crsp.{d_tab}
        WHERE permno IN ({perm_str})
          AND {dc['date']} BETWEEN '{global_start}' AND '{global_end}'
    """, date_cols=['date']))
dsf_raw = pd.concat(dsf_chunks, ignore_index=True)
dsf_raw['permno'] = dsf_raw['permno'].astype(int)
dsf_raw = dsf_raw.sort_values(['permno', 'date']).drop_duplicates(subset=['permno', 'date'])
for c in ['ret', 'prc', 'vol', 'shrout']:
    dsf_raw[c] = as_float(dsf_raw[c])
print(f'rows {len(dsf_raw):,} | last date {dsf_raw["date"].max().date()} | ret non-null {dsf_raw["ret"].notna().mean():.1%}')

Approximately 110257376 rows in crsp.dsf_v2.
crsp.dsf_v2: {'date': 'dlycaldt', 'ret': 'dlyret', 'prc': 'dlyprc', 'vol': 'dlyvol', 'shrout': 'shrout'}
Pulling crsp.dsf_v2 | 1,499 permnos | 2017-01-22 -> 2026-05-02
rows 3,182,801 | last date 2025-12-31 | ret non-null 99.9%


In [19]:
# ── 12c. Daily market index (CIZ, CRSP VW = indno 1000200; fallback Fama-French) ─
ic_d = pick_cols('crsp', 'inddlyseriesdata', {'date': ['dlycaldt'], 'ret': ['dlytotret', 'dlyret']})
if ic_d['date'] and ic_d['ret']:
    dsi = conn.raw_sql(f"""
        SELECT {ic_d['date']} AS date, {ic_d['ret']} AS mkt_ret
        FROM crsp.inddlyseriesdata
        WHERE indno = 1000200 AND {ic_d['date']} BETWEEN '{global_start}' AND '{global_end}'
    """, date_cols=['date'])
    src = 'CRSP VW (CIZ)'
else:
    dsi = conn.raw_sql(f"""
        SELECT date, mktrf + rf AS mkt_ret FROM ff.factors_daily
        WHERE date BETWEEN '{global_start}' AND '{global_end}'
    """, date_cols=['date'])
    src = 'Fama-French Mkt (mktrf + rf)'
dsi = dsi.sort_values('date').drop_duplicates('date').reset_index(drop=True)
dsi['mkt_ret'] = as_float(dsi['mkt_ret'])
dsi['mkt_ret_ew'] = np.nan
print(f'Market index: {src} | {len(dsi):,} days | {dsi["date"].min().date()} -> {dsi["date"].max().date()}')

Approximately 81957 rows in crsp.inddlyseriesdata.
crsp.inddlyseriesdata: {'date': 'dlycaldt', 'ret': 'dlytotret'}
Market index: CRSP VW (CIZ) | 2,249 days | 2017-01-23 -> 2025-12-31


In [20]:
# ── 12d.  Construct event-time indices ────────────────────────────────────────
#
# Step 1 – Expand dsf_raw into per-event rows by merging with the event panel.
#          A single permno can appear in multiple events (one per fiscal year),
#          so each stock-date row may fan out to multiple event rows.
#
# Step 2 – Filter each event's rows to its calendar-buffer window.
#
# Step 3 – Merge in the market benchmark by date.
#
# Step 4 – Identify the event anchor (t = 0):
#          RULE: the first trading day ON OR AFTER the filing_date.
#          Rationale: 10-K filings are often made after market close or on
#          weekends; the first subsequent trading day is when markets can first
#          react.  This rule is documented here and must be applied consistently
#          in all downstream event-study notebooks.
#
# Step 5 – Compute trading_day_rel:
#          Sequential trading-day index within each firm-year event window,
#          re-centred so that event_anchor → trading_day_rel = 0.
#
# Step 6 – Trim to the exact target window [−EVENT_PRE_DAYS, +EVENT_POST_DAYS].

# ── Step 1 & 2 : fan-out + calendar filter ──────────────────────────────────
events_all["_win_start"] = events_all["filing_date"] - pd.Timedelta(days=CAL_BUFFER_PRE)
events_all["_win_end"]   = events_all["filing_date"] + pd.Timedelta(days=CAL_BUFFER_POST)

dsf_panel = dsf_raw.merge(
    events_all[["permno", "cik", "gvkey", "fiscal_year",
                "filing_date", "_win_start", "_win_end"]],
    on="permno",
    how="inner",
)
dsf_panel = dsf_panel[
    (dsf_panel["date"] >= dsf_panel["_win_start"])
    & (dsf_panel["date"] <= dsf_panel["_win_end"])
].drop(columns=["_win_start", "_win_end"])

# ── Step 3 : merge market benchmark ─────────────────────────────────────────
dsf_panel = dsf_panel.merge(dsi[["date", "mkt_ret", "mkt_ret_ew"]], on="date", how="left")

# ── Step 4 : event anchor = first trading day on or after filing_date ────────
event_anchor_df = (
    dsf_panel[dsf_panel["date"] >= dsf_panel["filing_date"]]
    .groupby(["permno", "cik", "fiscal_year"])["date"]
    .min()
    .reset_index()
    .rename(columns={"date": "event_anchor"})
)
dsf_panel = dsf_panel.merge(
    event_anchor_df, on=["permno", "cik", "fiscal_year"], how="left"
)

# ── Step 5 : calendar_day_rel & trading_day_rel ──────────────────────────────
dsf_panel["calendar_day_rel"] = (dsf_panel["date"] - dsf_panel["filing_date"]).dt.days

dsf_panel = dsf_panel.sort_values(
    ["permno", "cik", "fiscal_year", "date"]
).reset_index(drop=True)

# Sequential trading-day rank within each event (0-based cumcount)
dsf_panel["_td_rank"] = dsf_panel.groupby(
    ["permno", "cik", "fiscal_year"]
).cumcount()

# Rank of the event anchor within each event window
anchor_rank_df = (
    dsf_panel[dsf_panel["date"] == dsf_panel["event_anchor"]]
    .groupby(["permno", "cik", "fiscal_year"])["_td_rank"]
    .first()
    .reset_index()
    .rename(columns={"_td_rank": "_anchor_rank"})
)
dsf_panel = dsf_panel.merge(
    anchor_rank_df, on=["permno", "cik", "fiscal_year"], how="left"
)
dsf_panel["trading_day_rel"] = (
    dsf_panel["_td_rank"] - dsf_panel["_anchor_rank"]
)
dsf_panel = dsf_panel.drop(columns=["_td_rank", "_anchor_rank"])

# ── Step 6 : trim to target trading-day window ───────────────────────────────
dsf_final = dsf_panel[
    (dsf_panel["trading_day_rel"] >= -EVENT_PRE_DAYS)
    & (dsf_panel["trading_day_rel"] <= EVENT_POST_DAYS)
].copy()

# ── Tidy column order ────────────────────────────────────────────────────────
col_order = [
    "permno", "gvkey", "cik", "fiscal_year",
    "filing_date", "event_anchor",
    "date", "calendar_day_rel", "trading_day_rel",
    "ret", "mkt_ret", "mkt_ret_ew",
    "prc", "vol", "shrout",
]
dsf_final = (
    dsf_final[[c for c in col_order if c in dsf_final.columns]]
    .reset_index(drop=True)
)

print(f"Event-window panel assembled")
print(f"  Rows                     : {len(dsf_final):,}")
print(f"  Firm-year events covered : {dsf_final.groupby(['cik','fiscal_year']).ngroups:,}")
print(f"  trading_day_rel range    : "
      f"{dsf_final['trading_day_rel'].min()} to {dsf_final['trading_day_rel'].max()}")

Event-window panel assembled
  Rows                     : 2,827,623
  Firm-year events covered : 9,774
  trading_day_rel range    : -260.0 to 30.0


In [21]:
# ── 12e.  Validation & summary statistics ─────────────────────────────────────

n_events_input   = n_with_permno
n_events_in_data = dsf_final.groupby(["cik", "fiscal_year"]).ngroups
n_td0_events     = (
    dsf_final[dsf_final["trading_day_rel"] == 0]
    .groupby(["cik", "fiscal_year"])
    .ngroups
)
avg_obs = dsf_final.groupby(["cik", "fiscal_year"]).size().mean()

print("=" * 60)
print("  Event-Window Coverage Summary")
print("=" * 60)
print(f"  Firm-year events in input (permno valid)  : {n_events_input:>6,}")
print(f"  Events with ≥1 daily return in window     : {n_events_in_data:>6,}"
      f"  ({n_events_in_data / n_events_input:.1%})")
print(f"  Events with trading_day_rel = 0 present   : {n_td0_events:>6,}"
      f"  ({n_td0_events / n_events_input:.1%})")
print(f"  Avg daily obs per event window            : {avg_obs:>9.1f}")
print()
print(f"  Dataset shape                             : {dsf_final.shape}")
print(f"  Distinct firm-year combinations           : "
      f"{dsf_final.groupby(['cik','fiscal_year']).ngroups:,}")
print(f"  trading_day_rel  min / max                : "
      f"{dsf_final['trading_day_rel'].min()} / {dsf_final['trading_day_rel'].max()}")
print()
print("  Return coverage:")
print(f"    ret (stock, non-null)    : {dsf_final['ret'].notna().mean():.1%}")
print(f"    mkt_ret (VW, non-null)   : {dsf_final['mkt_ret'].notna().mean():.1%}")
print(f"    mkt_ret_ew (EW, non-null): {dsf_final['mkt_ret_ew'].notna().mean():.1%}")

# ── Sample rows for 3 events around their event anchor ──────────────────────
print()
print("─" * 60)
print("  Sample: trading days [−3, +3] for 3 example events")
print("─" * 60)

sample_events = (
    dsf_final[["permno", "cik", "fiscal_year", "filing_date", "event_anchor"]]
    .drop_duplicates()
    .head(3)
)

display_cols = ["date", "calendar_day_rel", "trading_day_rel", "ret", "mkt_ret", "prc"]

for _, row in sample_events.iterrows():
    anchor_str = (row["event_anchor"].date()
                  if pd.notna(row["event_anchor"]) else "NaT")
    print(f"\n  permno={int(row['permno'])}  cik={int(row['cik'])}"
          f"  FY={int(row['fiscal_year'])}"
          f"  filing={row['filing_date'].date()}  anchor={anchor_str}")
    mask = (
        (dsf_final["permno"]      == row["permno"])
        & (dsf_final["cik"]       == row["cik"])
        & (dsf_final["fiscal_year"] == row["fiscal_year"])
        & (dsf_final["trading_day_rel"].between(-3, 3))
    )
    print(
        dsf_final.loc[mask, display_cols]
        .to_string(index=False, float_format=lambda x: f"{x:+.4f}")
    )

  Event-Window Coverage Summary
  Firm-year events in input (permno valid)  : 10,915
  Events with ≥1 daily return in window     :  9,774  (89.5%)
  Events with trading_day_rel = 0 present   :  9,774  (89.5%)
  Avg daily obs per event window            :     289.3

  Dataset shape                             : (2827623, 15)
  Distinct firm-year combinations           : 9,774
  trading_day_rel  min / max                : -260.0 / 30.0

  Return coverage:
    ret (stock, non-null)    : 100.0%
    mkt_ret (VW, non-null)   : 100.0%
    mkt_ret_ew (EW, non-null): 0.0%

────────────────────────────────────────────────────────────
  Sample: trading days [−3, +3] for 3 example events
────────────────────────────────────────────────────────────

  permno=10026  cik=785956  FY=2018  filing=2018-11-28  anchor=2018-11-28
      date  calendar_day_rel  trading_day_rel     ret  mkt_ret       prc
2018-11-23                -5          -3.0000 +0.0113  -0.0056 +153.8700
2018-11-26                -2     

In [22]:
# ── 12f.  Save event-window daily return panel ────────────────────────────────
#
# Output columns:
#   permno           – CRSP permanent security identifier
#   gvkey            – Compustat global company key
#   cik              – SEC CIK
#   fiscal_year      – Fiscal year of the 10-K filing
#   filing_date      – Actual date the 10-K was filed with the SEC
#   event_anchor     – First trading day on or after filing_date  (t = 0)
#   date             – Calendar date of this observation
#   calendar_day_rel – (date − filing_date).days  (signed integer)
#   trading_day_rel  – Trading-day index relative to event_anchor (0 = anchor)
#   ret              – CRSP daily holding-period return
#   mkt_ret          – CRSP VW market return (vwretd) — benchmark for CARs
#   mkt_ret_ew       – CRSP EW market return (ewretd) — robustness benchmark
#   prc              – Daily closing price (ABS; negative = mid-quote in CRSP)
#   vol              – Daily trading volume (shares)
#   shrout           – Shares outstanding (thousands)

dsf_final.to_parquet(CRSP_DSF_OUTPUT, index=False, compression="zstd")
file_mb = CRSP_DSF_OUTPUT.stat().st_size / 1e6

print(f"Saved → {CRSP_DSF_OUTPUT}")
print(f"  Rows     : {len(dsf_final):,}")
print(f"  Columns  : {list(dsf_final.columns)}")
print(f"  Size     : {file_mb:.1f} MB")
print()
print("Ready for downstream event-study notebooks.")
print("  Next steps: market-model estimation on pre-event window,")
print("              CAR construction, and realized-volatility measures.")

Saved → ..\data\intermediate\crsp_daily_returns_around_filing.parquet
  Rows     : 2,827,623
  Columns  : ['permno', 'gvkey', 'cik', 'fiscal_year', 'filing_date', 'event_anchor', 'date', 'calendar_day_rel', 'trading_day_rel', 'ret', 'mkt_ret', 'mkt_ret_ew', 'prc', 'vol', 'shrout']
  Size     : 29.7 MB

Ready for downstream event-study notebooks.
  Next steps: market-model estimation on pre-event window,
              CAR construction, and realized-volatility measures.


In [23]:
# ── 12g. Q4 earnings announcement dates (comp.fundq.rdq) ──────────────────────
gv = events_all['gvkey'].astype(str).str.split('.').str[0].str.zfill(6).unique().tolist()
gv_sql = ','.join(f"'{g}'" for g in gv)
rdq = conn.raw_sql(f"""
    SELECT gvkey, datadate, rdq
    FROM comp.fundq
    WHERE fqtr = 4 AND indfmt = 'INDL' AND datafmt = 'STD' AND popsrc = 'D' AND consol = 'C'
      AND datadate BETWEEN '2017-06-01' AND '2026-06-30'
      AND gvkey IN ({gv_sql})
""", date_cols=['datadate', 'rdq'])
rdq = rdq.dropna(subset=['rdq']).drop_duplicates(['gvkey', 'datadate'])
rdq.to_parquet(DATA_INTER / 'rdq_q4.parquet', index=False)
print(f'rdq rows: {len(rdq):,} | firms: {rdq.gvkey.nunique():,}')

rdq rows: 13,006 | firms: 1,485
